# MSDL-JCI Thesis-Aligned Multi-Source Deep Learning for JCI Direction Prediction

Implements the hybrid architecture from the thesis defense: LSTM (technical) + MLP Encoder (macro) + 
Frozen IndoBERT (news) + Adaptive MLP Fusion (Soft Gating Network). 
Walk-forward validation with AUC/F1/Sharpe metrics. Leak-free pipeline preserved.
Leak-free order: align â†’ train-only scaling â†’ window (lookback 28) â†’ t+5 labels.


In [ ]:
import warnings

warnings.filterwarnings("ignore", message=".*dropout option adds dropout.*")
import hashlib
import os
import random
import sqlite3
import sys
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, roc_curve
from torch import nn
from transformers import AutoModel, AutoTokenizer

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DB_PATH = Path(os.getenv("MSDL_DB", ROOT / "database" / "main_database.db"))
if not DB_PATH.is_absolute():
    DB_PATH = ROOT / DB_PATH
DB_PATH = DB_PATH.resolve()
sys.path.insert(0, str(ROOT))  # so `scripts.msdl_dates` imports regardless of the kernel's cwd


@dataclass(frozen=True)
class Config:
    look_back: int = 28
    horizon: int = 5
    hidden_layers: tuple = (32, 16)
    projection_dim: int = 64
    lstm_dim: int = 64
    fusion_output_dim: int = 3
    seed: int = 42
    tech_cols: list = field(default_factory=lambda: ["Close", "Volume", "RSI_14", "MACD", "MACD_Signal", "ATR_14", "SMA_20"])
    macro_cols: list = field(default_factory=lambda: ["bi_rate", "inflation_rate", "usd_idr"])


CFG = Config()


# --- Visualization helpers (DRY) ---
def _fmt_ax(ax, title, xlabel=None, ylabel=None, ylim=None):
    ax.set_title(title, fontsize=10, fontweight="bold")
    if xlabel: ax.set_xlabel(xlabel, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, fontsize=9)
    if ylim: ax.set_ylim(ylim)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    return ax

def _save_show():
    plt.tight_layout()
    plt.show()

def _bar_with_values(ax, x, y, fmt="{:.4f}", **kwargs):
    bars = ax.bar(x, y, **kwargs)
    for bar, val in zip(bars, y):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(), fmt.format(val),
                ha='center', va='bottom', fontsize=8)
    return bars


## Config & hyperparameters

One `Config` with Table 5 defaults baked in. `DB_PATH` defaults to the project-local `database/main_database.db`; set `MSDL_DB` only when using a different database file.

| Parameter | Default | Description |
|---|---|---|
| `look_back` | 28 | Technical lookback window (LSTM) |
| `horizon` | 5 | t+5 directional label |
| `seed` | 42 | RNG seed |
| `hidden_layers` | (32, 16) | MLP Encoder hidden units |
| `projection_dim` | 64 | News embedding projection (768 â†’ 64) |
| `lstm_dim` | 64 | LSTM hidden size |
| `fusion_input_dim` | 144 | Concatenated features (64 + 16 + 64) |
| `fusion_output_dim` | 3 | Gating weights (Î±, β, γ) via Softmax |

Run this notebook with a local Jupyter kernel from the project directory (or a subdirectory); the project root is located by walking upward to `pyproject.toml`.

## Entity contract + SQLite reader

Verbatim schema of `database/main_database.db`. Every read funnels through
`read_table`, which raises on schema drift. Tables: `jci_historical`, `bi_rate`,
`inflation_data`, `kurs_usdidr` + article tables (`cnbc/detik/kontan_ihsg_articles`).


In [ ]:
ENTITY_TABLES = {
    "jci_historical": ("Date", "Close", "High", "Low", "Open", "Volume"),
    "bi_rate": ("Period", "BI-7Day-RR", "publication_date"),
    "inflation_data": ("Periode", "Data Inflasi", "release_date"),
    "kurs_usdidr": ("Date", "Close", "High", "Low", "Open"),
    "cnbc_ihsg_articles": ("title", "url", "publish_date", "content", "scraped_at"),
    "detik_ihsg_articles": ("title", "url", "snippet", "published", "scraped_at"),
    "kontan_ihsg_articles": ("title", "url", "snippet", "published", "category", "scraped_at"),
}


def _connect_database():
    if not DB_PATH.is_file():
        raise FileNotFoundError(
            f"SQLite database not found: {DB_PATH}. Run Jupyter from this project, restore "
            "database/main_database.db, or set MSDL_DB to another existing database file."
        )
    try:
        return sqlite3.connect(DB_PATH)
    except sqlite3.OperationalError as exc:
        raise sqlite3.OperationalError(
            f"Could not open SQLite database at {DB_PATH}. Check that the runtime can read "
            "the file and access its parent directory. Check DB_PATH or MSDL_DB."
        ) from exc


def read_table(name, con=None):
    # ponytail: stdlib sqlite3, not SQLAlchemy — single-file DB, no server, no ORM gain.
    # Upgrade path: sqlalchemy.create_engine if this ever needs pools/concurrency.
    # con=None opens a short-lived connection; callers (load_frames/load_news) may pass
    # a shared connection to avoid reopening the DB on every table read (#68).
    own = con is None
    if own:
        con = _connect_database()
    try:
        df = pd.read_sql_query(f"SELECT * FROM [{name}]", con)
    finally:
        if own:
            con.close()
    missing = [c for c in ENTITY_TABLES[name] if c not in df.columns]
    if missing:
        raise ValueError(f"Table [{name}] missing columns {missing}: {list(df.columns)}")
    return df


print(f"source: SQLite {DB_PATH} ({'available' if DB_PATH.is_file() else 'missing'})")

## Utilities

Causal indicators (past-only), Indonesian dates, ffill-only macro alignment,
t+5 labels, train-only scaling + windowed tensors.

In [ ]:
import dataclasses

from sklearn.preprocessing import MinMaxScaler, StandardScaler


def parse_id_date(s, day_first=True):
    """Parse Indonesian-formatted dates using pandas mixed-format parser."""
    return pd.to_datetime(
        s.astype(str).str.lower(), format="mixed", errors="coerce", dayfirst=day_first
    )


def add_indicators(df):
    c = df["Close"].astype(float)
    h, l = df["High"].astype(float), df["Low"].astype(float)
    d = c.diff()
    ag = d.clip(lower=0).ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()
    al = (-d.clip(upper=0)).ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()
    df["RSI_14"] = 100 - 100 / (1 + ag / (al + 1e-9))
    df["MACD"] = c.ewm(span=12, adjust=False).mean() - c.ewm(span=26, adjust=False).mean()
    df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    pc = c.shift(1)
    tr = pd.concat([h - l, (h - pc).abs(), (l - pc).abs()], axis=1).max(axis=1)
    df["ATR_14"] = tr.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()
    df["SMA_20"] = c.rolling(20).mean()
    return df


def load_frames():
    """(jci, bi, inf, kur) raw frames from SQLite."""
    with _connect_database() as con:
        return (
            read_table("jci_historical", con),
            read_table("bi_rate", con),
            read_table("inflation_data", con),
            read_table("kurs_usdidr", con),
        )


def align(jci, bi, inf, kur, news_daily=None, horizon=None):
    horizon = CFG.horizon if horizon is None else int(horizon)
    if horizon < 1:
        raise ValueError("horizon must be a positive number of trading days")
    jci = jci.copy()
    jci["Date"] = pd.to_datetime(jci["Date"], format="mixed", errors="coerce")
    jci = jci.dropna(subset=["Date", "Close"]).sort_values("Date").reset_index(drop=True)
    add_indicators(jci)

    bi = bi.copy()
    bi["Date"] = parse_id_date(bi["Period"])
    bi["bi_rate"] = (
        bi["BI-7Day-RR"].astype(str).str.replace("%", "", regex=False).str.strip().astype(float)
    )
    bi = bi.dropna(subset=["Date"]).sort_values("Date")[["Date", "bi_rate"]]

    inf = inf.copy()
    inf["Date"] = pd.to_datetime(inf["release_date"], format="%Y-%m-%d", errors="coerce")
    inf["inflation_rate"] = (
        inf["Data Inflasi"].astype(str).str.replace("%", "", regex=False).str.strip().astype(float)
    )
    inf = inf.dropna(subset=["Date"]).sort_values("Date")[["Date", "inflation_rate"]]

    kur = kur.copy()
    kur["Date"] = pd.to_datetime(kur["Date"], format="mixed", errors="coerce")
    kur["usd_idr"] = pd.to_numeric(
        kur["Close"].astype(str).str.replace(",", "", regex=False), errors="coerce"
    )
    kur = kur.dropna(subset=["Date"]).sort_values("Date")[["Date", "usd_idr"]]

    m = pd.merge(jci, kur, on="Date", how="left")
    m = pd.merge_asof(
        m.sort_values("Date"), bi.sort_values("Date"), on="Date", direction="backward"
    )
    m = pd.merge_asof(
        m.sort_values("Date"), inf.sort_values("Date"), on="Date", direction="backward"
    )
    for col in CFG.macro_cols:
        m[col] = m[col].ffill()
    m = m.dropna(subset=list(CFG.macro_cols)).reset_index(drop=True)

    fc = m["Close"].shift(-horizon)
    m["return_horizon"] = (fc - m["Close"]) / m["Close"]
    if horizon == 5:
        m["return_5d"] = m["return_horizon"]
    m["target_direction"] = (fc > m["Close"]).astype(float)
    m["future_close"] = fc
    m = attach_news(m, news_daily)  # past-only daily IndoBERT means, zeros where no news
    return m.dropna(subset=list(CFG.tech_cols) + ["future_close"]).reset_index(drop=True)


NEWS_COLS = [f"news_{i}" for i in range(768)]


def load_news():
    """All 3 portals, 'IHSG'-filtered corpus per thesis 2.1. Weekend articles -> next Monday."""
    from scripts.msdl_dates import clean_day  # single copy; audit scripts import the same one
    specs = [  # (table, date col, title col, body col, tag)
        ("detik_ihsg_articles", "published", "title", "snippet", "detik"),
        ("cnbc_ihsg_articles", "publish_date", "title", "content", "cnbc"),
        ("kontan_ihsg_articles", "published", "title", "snippet", "kontan"),
    ]
    frames = []
    with _connect_database() as con:
        for table, dcol, tcol, bcol, tag in specs:
            d = read_table(table, con)
            frames.append(pd.DataFrame({
                "Date": pd.to_datetime(d[dcol].astype(str).map(clean_day), format="mixed", errors="coerce", dayfirst=True),
                "text": (d[tcol].fillna("") + " " + d[bcol].fillna("")).str.strip(), "src": tag}))
    out = pd.concat(frames, ignore_index=True)
    out = out.dropna(subset=["Date"])
    out = out[out["text"] != ""].sort_values("Date").reset_index(drop=True)
    # thesis 2.3: Sat/Sun articles accumulate to next Monday
    wd = out["Date"].dt.weekday
    out["Date"] = out["Date"] + pd.to_timedelta(((7 - wd) % 7).where(wd >= 5, 0), unit="D")
    return out


def news_embeddings(texts, cache_path):
    """Frozen IndoBERT CLS per article, cached on disk. Returns (n, 768). Offline-safe."""
    import torch
    keys = [hashlib.md5(t.encode()).hexdigest() for t in texts]
    cache = {}
    cp = Path(cache_path)
    if cp.exists():
        z = np.load(cp, allow_pickle=True)
        cache = {k: v for k, v in zip(z["keys"], z["vecs"])}
    missing = [t for t, k in zip(texts, keys) if k not in cache]
    if missing:
        # TRANSFORMERS_OFFLINE=1 in CI prevents download; force local_files_only=True
        try:
            tok = AutoTokenizer.from_pretrained(
                "indobenchmark/indobert-base-p1", local_files_only=True, trust_remote_code=False
            )
            bert = AutoModel.from_pretrained(
                "indobenchmark/indobert-base-p1", local_files_only=True, trust_remote_code=False
            )
        except (OSError, ValueError) as e:
            # Crash in CI with TRANSFORMERS_OFFLINE=1 if no cached model
            if os.environ.get("TRANSFORMERS_OFFLINE", "0") == "1":
                raise RuntimeError(
                    "TRANSFORMERS_OFFLINE=1 but no cached IndoBERT model found. "
                    "Pre-download the model or remove TRANSFORMERS_OFFLINE."
                ) from e
            # Local development: allow download (warn about offline intent)
            import warnings
            warnings.warn(
                "No cached IndoBERT model found. Attempting download. "
                "Set TRANSFORMERS_OFFLINE=1 for CI reproducibility."
            )
            tok = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
            bert = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")
        bert.eval()
        for i in range(0, len(missing), 32):
            batch = missing[i : i + 32]
            enc = tok(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
            with torch.no_grad():
                h = bert(**enc).last_hidden_state[:, 0, :].cpu().numpy()
            for t, v in zip(batch, h):
                cache[hashlib.md5(t.encode()).hexdigest()] = v
        cp.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(cp, keys=np.array(list(cache)), vecs=np.stack(list(cache.values())))
    return np.stack([cache[k] for k in keys]).astype(np.float32)


def attach_news(m, news_daily):
    """Per trading day: mean article vector strictly BEFORE that date (no same-day leak). Vectorized."""
    emb = np.zeros((len(m), 768), dtype=np.float32)
    if news_daily is not None and len(news_daily):
        vecs = np.stack(news_daily["vec"].values)
        days = news_daily["day"].values
        # days must be sorted for searchsorted; load_news returns sorted
        idx = np.searchsorted(days, m["Date"].values, side="left")
        cumsum = np.cumsum(vecs, axis=0)  # (M, 768)
        emb[idx > 0] = cumsum[idx[idx > 0] - 1] / idx[idx > 0, None]
    m = pd.concat([m.copy(), pd.DataFrame(emb, columns=NEWS_COLS, index=m.index)], axis=1)
    return m


def make_tensors(df, train_end_idx, news_agg="mean", cfg=None):
    cfg = cfg or CFG
    assert 0 < train_end_idx <= len(df), (
        f"train_end_idx {train_end_idx} out of bounds [1, {len(df)}]"
    )
    raw_tech = df[cfg.tech_cols].values.astype(np.float32)
    raw_macro = df[cfg.macro_cols].values.astype(np.float32)

    fs = MinMaxScaler()
    fs.fit(raw_tech[:train_end_idx])
    ms = StandardScaler()  # thesis 2.2: macro uses Z-score, fitted on train only
    ms.fit(raw_macro[:train_end_idx])
    st, sm = fs.transform(raw_tech), ms.transform(raw_macro)
    lb = cfg.look_back
    X_tech = np.stack([st[i - lb + 1 : i + 1] for i in range(lb - 1, len(df))])
    raw_daily_news = df[NEWS_COLS].values.astype(np.float32)
    agg = (lambda w: w.mean(axis=0)) if news_agg == "mean" else (lambda w: w[-1])
    X_news = np.stack([agg(raw_daily_news[i - lb + 1 : i + 1]) for i in range(lb - 1, len(df))])
    idx = np.arange(lb - 1, len(df))
    return (
        X_tech.astype(np.float32),
        sm[idx],
        X_news,
        df["target_direction"].values[idx].astype(np.float32),
        df["Date"].dt.strftime("%Y-%m-%d").values[idx],
        fs,
    )


# ====== THESIS MODEL ARCHITECTURE (Table 5) ======


class StockModel(nn.Module):
    """LSTM (tech) + MLP (macro) + Frozen IndoBERT (news) + Adaptive MLP Fusion (Soft Gating)."""

    VARIANTS = ("full", "lstm", "lstm_macro", "lstm_news", "static")  # thesis 2.6 ablation

    def __init__(self, cfg, variant="full"):
        super().__init__()
        assert variant in self.VARIANTS, variant
        self.variant = variant
        self.lstm = nn.LSTM(len(cfg.tech_cols), cfg.lstm_dim, 1, batch_first=True)  # no dropout arg
        self.mlp = nn.Sequential(
            nn.Linear(len(cfg.macro_cols), 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU()
        )
        self.macro_proj = nn.Linear(16, cfg.lstm_dim)
        self.news_proj = nn.Linear(768, cfg.lstm_dim)
        self.gate = nn.Linear(cfg.lstm_dim + 16 + cfg.lstm_dim, cfg.fusion_output_dim)
        self.head = nn.Linear(cfg.lstm_dim, 1)
        self.head_tm = nn.Linear(cfg.lstm_dim + 16, 1)
        self.head_tn = nn.Linear(cfg.lstm_dim + cfg.lstm_dim, 1)
        self.head_t = nn.Linear(cfg.lstm_dim, 1)
        self.criterion = nn.BCEWithLogitsLoss()

    def forward(self, tech_x, macro_x, news_vec, targets=None):
        _, (hn, _) = self.lstm(tech_x)
        tech_feat = hn.squeeze(0)
        macro_raw = self.mlp(macro_x)
        macro_feat = self.macro_proj(macro_raw)
        news_feat = self.news_proj(news_vec)  # frozen IndoBERT vecs precomputed outside
        combined = torch.cat([tech_feat, macro_raw, news_feat], dim=1)
        weights = torch.softmax(self.gate(combined), dim=1)
        gated = (
            weights[:, 0:1] * tech_feat + weights[:, 1:2] * macro_feat + weights[:, 2:3] * news_feat
        )
        if self.variant == "lstm":
            logits = self.head_t(tech_feat).squeeze(-1)
            weights = torch.zeros_like(weights)
        elif self.variant == "lstm_macro":
            logits = self.head_tm(torch.cat([tech_feat, macro_raw], dim=1)).squeeze(-1)
            weights = torch.zeros_like(weights)
        elif self.variant == "lstm_news":
            logits = self.head_tn(torch.cat([tech_feat, news_feat], dim=1)).squeeze(-1)
            weights = torch.zeros_like(weights)
        elif self.variant == "static":
            logits = self.head((tech_feat + macro_feat + news_feat) / 3).squeeze(-1)
            weights = torch.full_like(weights, 1 / 3)
        else:
            logits = self.head(gated).squeeze(-1)
        if targets is not None:
            return logits, self.criterion(logits, targets), weights
        return logits, None, weights





class FocalLoss(nn.Module):
    """BCE focal variant (Lin et al. 2017): down-weights easy majority-class samples."""

    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma

    def forward(self, logits, targets):
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt = torch.exp(-bce)
        return ((self.alpha * targets + (1 - self.alpha) * (1 - targets)) * (1 - pt) ** self.gamma * bce).mean()


def branch_grad_norms(m):
    """E4 audit: gradient mass per branch after one backward pass."""
    out = {}
    for k, ps in {"lstm": m.lstm.parameters(), "mlp": list(m.mlp.parameters()) + list(m.macro_proj.parameters()),
            "news": m.news_proj.parameters(), "gate+head": list(m.gate.parameters()) + list(m.head.parameters())}.items():
        gs = [p.grad.flatten() for p in ps if p.grad is not None]
        out[k] = float(torch.cat(gs).norm()) if gs else 0.0  # 0.0 = branch unused by variant
    return out


def train_fold(Xt_tech, Xt_macro, Xt_news, yt, cfg, variant="full", max_epochs=100, patience=8, loss="bce", lr=1e-3):
    """Fresh model per fold, Adam + early stopping on last-10% val split."""
    torch.manual_seed(cfg.seed)
    m = StockModel(cfg, variant)
    pos = float(yt.mean())
    m.criterion = (torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1 - pos) / max(pos, 1e-6))) if loss == "bce"
        else FocalLoss(alpha=1 - pos))
    m.train()
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    T = lambda a: torch.tensor(a, dtype=torch.float32)
    n, nv = len(yt), max(1, len(yt) // 10)
    tr = (slice(None, n - nv), slice(n - nv, None))
    Xv = (T(Xt_tech[tr[1]]), T(Xt_macro[tr[1]]), T(Xt_news[tr[1]]), T(yt[tr[1]]))
    best, best_state, bad = float("inf"), None, 0
    hist = {"train": [], "val": []}  # E2: learning curves
    for ep in range(max_epochs):
        opt.zero_grad()
        _, loss, _ = m(T(Xt_tech[tr[0]]), T(Xt_macro[tr[0]]), T(Xt_news[tr[0]]), T(yt[tr[0]]))
        loss.backward()
        if ep == 0:
            g0 = branch_grad_norms(m)  # E4: gradients alive at init?
        opt.step()
        m.eval()
        with torch.no_grad():
            _, vl, _ = m(*Xv)
        m.train()
        hist["train"].append(loss.item())
        hist["val"].append(vl.item())
        if vl.item() < best - 1e-4:
            best, best_state, bad = vl.item(), {k: v.cpu().clone() for k, v in m.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                break
    m.load_state_dict(best_state)
    m.eval()
    m.hist_ = {"epochs_run": ep + 1, "train": hist["train"], "val": hist["val"], "grad0": g0}
    return m


def trading_stats(dates, probs, df, thr=0.5, hold=5, cost_bps=0.0):
    """Long above `thr`, short otherwise; each signal earns the next `hold` daily returns.
    Strategy and buy-hold both use daily compounding over the test window."""
    px = df.set_index(df["Date"].dt.strftime("%Y-%m-%d"))
    d1 = px["Close"].pct_change().fillna(0.0)
    sig = pd.Series(0.0, index=d1.index)
    for d, v in zip(dates, np.where(np.array(probs) > thr, 1.0, -1.0)):
        if d in sig.index:
            sig.loc[d] += v
    active = sig.rolling(hold, min_periods=1).sum()  # open long/short signals in [t-hold+1, t], averaged by hold
    position = active.shift(1).fillna(0.0) / hold  # position at s reflects signals in [s-hold, s-1]
    pr = (d1 * position).values
    if cost_bps:
        pr = pr - (position.diff().fillna(position).abs() * (cost_bps / 10000.0)).values  # one-way cost per unit exposure change
    n_trades = int((position.diff().fillna(position).abs() > 0).sum())
    bh = d1.values  # buy-hold: daily compounding (matches strategy basis)
    lo, hi = min(dates), max(dates)  # report both series over the test window only
    out = {}
    for name, x in (("strategy", pr), ("buyhold", bh)):
        x = pd.Series(x, index=d1.index).loc[lo:hi].values
        x = x[np.isfinite(x)]
        cum = np.cumprod(1 + x)
        peak = np.maximum.accumulate(cum)
        dd = np.min(cum / np.maximum(peak, 1e-9) - 1)
        out[name] = {
            "daily": [float(v) for v in x],
            "equity": [float(v) for v in cum],
            "total_return": float(cum[-1] - 1),
            "sharpe": float(x.mean() / (x.std() + 1e-9) * np.sqrt(252)),
            "max_drawdown": float(dd),
            "win_rate": float((x > 0).mean()),
        }
    out["strategy"]["n_trades"] = n_trades
    return out


def youden_threshold(yt, pt):
    """(1) best train threshold by Youden's J; falls back to 0.5 on single-class trains."""
    if len(set(yt.tolist())) < 2:
        return 0.5
    fpr, tpr, thr = roc_curve(yt, pt)
    return float(thr[int(np.argmax(tpr - fpr))])


def paired_stats(a, b):
    """(3) paired mean-diff with 95% CI using scipy.stats.t (robust to any sample size)."""
    from scipy.stats import t as t_dist
    d = np.array(a) - np.array(b)
    n = len(d)
    se = d.std(ddof=1) / np.sqrt(n)
    t_val = t_dist.ppf(0.975, df=n - 1) if n > 1 else 12.706
    return float(d.mean()), float(d.mean() - t_val * se), float(d.mean() + t_val * se)


def classical_baseline(df, cfg, n_splits=5, model="logreg"):
    """Sklearn baseline (logreg/rf) on the same leak-free walk-forward splits.
    Same make_tensors inputs (train-prefix scalers), same Youden-J protocol.
    ponytail: defaults only (C=1.0, 300 trees) â€” tuning it would invite overfitting the null."""
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import TimeSeriesSplit
    from sklearn.preprocessing import StandardScaler
    lb = cfg.look_back
    n_win = len(df) - lb + 1
    tss = TimeSeriesSplit(n_splits=n_splits, gap=lb + cfg.horizon)  # purge lookback + horizon: train never touches test prices
    fold_metrics, P, T, D = [], [], [], []
    for i, (tr, te) in enumerate(tss.split(np.arange(n_win))):
        fit_end = int(tr.max() + lb)  # identical train prefix to deep path
        Xa, Xb, Xc, ya, dts, _ = make_tensors(df, fit_end, "mean", cfg)
        F = lambda A: A.reshape(len(A), -1)
        Xa_tr, Xa_te = F(Xa[tr]), F(Xa[te])
        Xb_tr, Xb_te = Xb[tr], Xb[te]
        Xc_tr, Xc_te = Xc[tr], Xc[te]
        # Scale tech+macro together; scale news separately on train (matches deep path)
        Xtm_tr = np.concatenate([Xa_tr, Xb_tr], axis=1)
        Xtm_te = np.concatenate([Xa_te, Xb_te], axis=1)
        sc_tm = StandardScaler().fit(Xtm_tr)
        sc_news = StandardScaler().fit(Xc_tr)
        Xtr = np.concatenate([sc_tm.transform(Xtm_tr), sc_news.transform(Xc_tr)], axis=1)
        Xte = np.concatenate([sc_tm.transform(Xtm_te), sc_news.transform(Xc_te)], axis=1)
        clf = (LogisticRegression(max_iter=2000) if model == "logreg"
               else RandomForestClassifier(n_estimators=300, random_state=cfg.seed, n_jobs=-1))
        Ztr, Zte = Xtr, Xte
        clf.fit(Ztr, ya[tr])
        p = clf.predict_proba(Zte)[:, 1]
        thr = youden_threshold(ya[tr], clf.predict_proba(Ztr)[:, 1])
        pred = (p > thr).astype(float)
        yte = ya[te]
        P.extend(p.tolist()); T.extend(yte.tolist()); D.extend(dts[te].tolist())
        f1c = f1_score(yte, pred, average=None, labels=[0, 1], zero_division=0)
        fold_metrics.append({"fold": i, "auc": float(roc_auc_score(yte, p)), "thr": thr,
            "acc": float(accuracy_score(yte, pred)), "f1": float(f1_score(yte, pred, zero_division=0)),
            "f1_down": float(f1c[0]), "f1_up": float(f1c[1]), "n_samples": len(te)})
    P = np.array(P)
    all05 = (P > 0.5).astype(float)
    f1a = f1_score(T, all05, average=None, labels=[0, 1], zero_division=0)
    return {"auc": float(roc_auc_score(T, P)), "acc": float(accuracy_score(T, all05)),
        "f1": float(f1_score(T, all05, zero_division=0)),
        "f1_down": float(f1a[0]), "f1_up": float(f1a[1]),
        "fold_metrics": fold_metrics, "preds": P.tolist(), "truths": T,
        "test_dates": [str(d) for d in D],
        "trading": trading_stats(D, P.tolist(), df)}


def walk_forward_evaluate(df, cfg, n_splits=5, variants=("full",), news_agg="mean", loss="bce", lr=1e-3, look_back=None, return_details=False):
    from sklearn.model_selection import TimeSeriesSplit
    cfg = dataclasses.replace(cfg, look_back=look_back or cfg.look_back)
    lb = cfg.look_back
    n_win = len(df) - lb + 1
    tss = TimeSeriesSplit(n_splits=n_splits, gap=lb + cfg.horizon)  # purge lookback + horizon: train never touches test prices
    results = {}
    for variant in variants:
        base = variant.removesuffix("_shuffled")  # E1 control
        fold_metrics, weights_list, preds_all, truths_all, dates_all, curves, grads = [], [], [], [], [], [], []
        fold_trading = []
        for i, (tr, te) in enumerate(tss.split(np.arange(n_win))):
            fit_end = int(tr.max() + lb)  # df rows informing train samples only
            Xa, Xb, Xc, ya, dts, _ = make_tensors(df, fit_end, news_agg, cfg)
            Xn_tr = Xc[tr]
            if variant.endswith("_shuffled"):  # E1: break news-label link, keep marginals
                Xn_tr = Xn_tr[np.random.default_rng(cfg.seed + i).permutation(len(Xn_tr))]
            model = train_fold(Xa[tr], Xb[tr], Xn_tr, ya[tr], cfg, base, loss=loss, lr=lr)
            curves.append(model.hist_)
            grads.append(model.hist_["grad0"])
            with torch.no_grad():
                T = lambda a: torch.tensor(a, dtype=torch.float32)
                logits, _, w = model(T(Xa[te]), T(Xb[te]), T(Xc[te]))
                prob = torch.sigmoid(logits).cpu().numpy()
                tlog, _, _ = model(T(Xa[tr]), T(Xb[tr]), T(Xn_tr))
                thr = youden_threshold(ya[tr], torch.sigmoid(tlog).cpu().numpy())  # (1) tuned on train only
                pred = (prob > thr).astype(float)
            yte = ya[te]
            truths_all.extend(yte.tolist())
            preds_all.extend(prob.tolist())
            dates_all.extend(dts[te].tolist())
            weights_list.append(w.mean(dim=0).cpu().numpy())
            au = float(roc_auc_score(yte, prob) if len(set(yte.tolist())) > 1 else 0.5)
            f1c = f1_score(yte, pred, average=None, labels=[0, 1], zero_division=0)
            fold_trading.append({"fold": i, **trading_stats(dts[te].tolist(), prob.tolist(), df)})
            fold_metrics.append({"fold": i, "auc": au, "thr": thr,
                "acc": float(accuracy_score(yte, pred)),
                "f1": float(f1_score(yte, pred, zero_division=0)),
                "f1_down": float(f1c[0]), "f1_up": float(f1c[1]), "n_samples": len(te),
                "preds": prob.tolist(), "truths": yte.tolist()})
        all_pred = [1 if x > 0.5 else 0 for x in preds_all]
        f1a = f1_score(truths_all, all_pred, average=None, labels=[0, 1], zero_division=0)
        results[variant] = {
            "auc": float(roc_auc_score(truths_all, preds_all)),
            "acc": float(accuracy_score(truths_all, all_pred)),
            "f1": float(f1_score(truths_all, all_pred, zero_division=0)),
            "f1_down": float(f1a[0]), "f1_up": float(f1a[1]),
            "fold_metrics": fold_metrics, "fold_trading": fold_trading, "weights_list": weights_list,
            "preds": preds_all, "truths": truths_all,
            "test_dates": [str(d) for d in dates_all],
            "fold_indices": list(range(n_splits)),
            "trading": trading_stats(dates_all, preds_all, df),
            "learning_curves": curves, "grad_norms": grads,
        }
    return results


## Run

Align â†’ train-only scaling on the 70% prefix â†’ windowed tensors.

**What to look at:** `news articles` (per-portal counts, must total 20,784), `articles on/before JCI end` (past-only proof), `pos_rate` (â‰ˆ0.55: labels slightly UP-leaning). Labels: `target=1` iff close(t+5) > close(t) â€” no future leak by construction.


In [ ]:
# --- Data preparation (run once, then re-use for all experiments) ---
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
random.seed(CFG.seed)
frames = load_frames()
jci_max = pd.to_datetime(frames[0]["Date"], format="mixed", errors="coerce").max()
news = load_news()
print(f"news articles: {len(news)}" + (f" {dict(news['src'].value_counts())}" if len(news) else ""))
if len(news):
    vecs = news_embeddings(news["text"].tolist(), ROOT / "database" / "news_emb_cache.npz")
    news["vec"] = list(vecs)
    news["day"] = pd.to_datetime(news["Date"].dt.date)
    print(f"news embedded: {vecs.shape}, range: {news['day'].min()} → {news['day'].max()}")
    print(f"articles on/before JCI end ({jci_max.date()}): {(news['day'] <= jci_max).sum()} (rest cannot inform t+5 labels)")
    news_daily = news[["day", "vec", "src"]]
else:
    news_daily = None
df = align(*frames, news_daily)
lb = CFG.look_back
n_win = len(df) - lb + 1
train_end_df = int(n_win * 0.70) + lb - 1
X_tech, X_macro, X_news, y, dates, feat_scaler = make_tensors(df, train_end_df)
print(f"rows: {len(df)} | samples: {len(y)} | X_tech: {X_tech.shape}")
print(f"range: {dates[0]} → {dates[-1]} | pos_rate: {y.mean():.3f}")

## Dataset Validation & Distribution

Full-sample descriptive diagnostics for technical, macroeconomic, and news modalities. Feature distributions are shown by the t+5 direction label, with raw-unit summaries, missingness, temporal coverage, and news-embedding diagnostics. These plots describe the dataset; they do not modify preprocessing or model inputs.

In [ ]:
# --- Multimodal dataset distribution audit (descriptive; no preprocessing or model fitting) ---
_sample_dates = pd.to_datetime(pd.Series(dates), errors="raise")
_base = df.copy()
_base["Date"] = pd.to_datetime(_base["Date"], errors="raise")
_labels = pd.DataFrame({"Date": _sample_dates, "target": np.asarray(y, dtype=int)})
_sample = _labels.merge(_base, on="Date", how="left", validate="one_to_one")
if len(_sample) != len(_labels) or _sample["target_direction"].isna().any():
    raise ValueError("Could not align every model sample with its prepared row and target.")

_class_names = {0: "DOWN", 1: "UP"}
_class_colors = {0: "#D95F5F", 1: "#2E9D72"}
_tech_cols = list(dict.fromkeys(CFG.tech_cols + ["ATR_14", "RSI_14", "MACD"]))
_feature_groups = {"Technical": _tech_cols, "Macro": list(CFG.macro_cols)}
_missing_cols = [c for cols in _feature_groups.values() for c in cols if c not in _sample.columns]
if _missing_cols:
    raise KeyError(f"Expected modality features are missing: {_missing_cols}")

# Plot raw aligned features. Volume is log-scaled for display only; summaries remain in original units.
for _modality, _cols in _feature_groups.items():
    _ncols = 3
    _nrows = int(np.ceil(len(_cols) / _ncols))
    _fig, _axes = plt.subplots(_nrows, _ncols, figsize=(15, 3.8 * _nrows), squeeze=False)
    for _i, _feature in enumerate(_cols):
        _ax = _axes.flat[_i]
        _raw = pd.to_numeric(_sample[_feature], errors="coerce")
        _finite = _raw[np.isfinite(_raw)]
        if _finite.empty:
            _ax.set_title(f"{_feature}: no finite values")
            _ax.axis("off")
            continue
        _log_volume = _feature == "Volume" and _finite.min() >= 0
        _pooled = np.log1p(_finite.to_numpy()) if _log_volume else _finite.to_numpy()
        _lo, _hi = np.quantile(_pooled, [0.005, 0.995])
        if not np.isfinite(_lo + _hi) or _lo >= _hi:
            _lo, _hi = float(np.min(_pooled)), float(np.max(_pooled))
        if _lo >= _hi:
            _lo, _hi = _lo - 0.5, _hi + 0.5
        _bins = np.linspace(_lo, _hi, 36)
        _tail_n = int(((_pooled < _lo) | (_pooled > _hi)).sum())
        for _cls in (0, 1):
            _values = pd.to_numeric(
                _sample.loc[_sample["target"] == _cls, _feature], errors="coerce"
            ).to_numpy(dtype=float)
            _values = _values[np.isfinite(_values)]
            if _log_volume:
                _values = np.log1p(_values)
            _shown = _values[(_values >= _lo) & (_values <= _hi)]
            if len(_shown):
                _ax.hist(_shown, bins=_bins, density=True, histtype="step", linewidth=1.8,
                         color=_class_colors[_cls],
                         label=f"{_class_names[_cls]} (n={len(_values):,})")
                _ax.axvline(np.median(_values), color=_class_colors[_cls], linestyle="--",
                            linewidth=1, alpha=0.85)
        _ax.set_title(_feature, loc="left", fontweight="bold")
        _ax.set_xlabel("log(1 + Volume)" if _log_volume else _feature)
        _ax.set_ylabel("Density")
        _ax.grid(axis="y", alpha=0.2)
        _ax.spines[["top", "right"]].set_visible(False)
        _bad_pct = 100 * (~np.isfinite(_raw)).mean()
        _ax.text(0.98, 0.96, f"n={len(_finite):,}; missing/nonfinite={_bad_pct:.1f}%",
                 transform=_ax.transAxes, ha="right", va="top", fontsize=7, color="#555555")
        if _tail_n:
            _ax.text(0.02, 0.96, f"{_tail_n:,} outside plotted P0.5-P99.5 range",
                     transform=_ax.transAxes, ha="left", va="top", fontsize=7, color="#555555")
    for _ax in _axes.flat[len(_cols):]:
        _ax.set_visible(False)
    _handles, _legend_labels = _axes.flat[0].get_legend_handles_labels()
    if _handles:
        _fig.legend(_handles, _legend_labels, loc="upper center", ncol=2,
                    bbox_to_anchor=(0.5, 1.01), frameon=False)
    _fig.suptitle(f"{_modality} feature distributions by t+5 direction",
                  fontsize=14, fontweight="bold", y=1.05)
    _fig.text(0.5, 0.005,
              "Density histograms use raw values; dashed lines mark class medians. Plots clip only at pooled P0.5-P99.5; summaries below retain all finite observations.",
              ha="center", fontsize=8, color="#555555")
    _fig.tight_layout(rect=[0, 0.035, 1, 0.94])
    plt.show()

# Per-feature, per-class statistics in original units, including explicit non-finite rates.
for _modality, _cols in _feature_groups.items():
    _rows = []
    for _feature in _cols:
        _series = pd.to_numeric(_sample[_feature], errors="coerce")
        for _cls in (0, 1):
            _class_values = _series.loc[_sample["target"] == _cls]
            _values = _class_values[np.isfinite(_class_values)]
            _rows.append({
                "feature": _feature, "class": _class_names[_cls], "n": len(_values),
                "missing_nonfinite_pct": 100 * (~np.isfinite(_class_values)).mean(),
                "mean": _values.mean(), "sd": _values.std(), "median": _values.median(),
                "q01": _values.quantile(0.01), "q05": _values.quantile(0.05),
                "q25": _values.quantile(0.25), "q75": _values.quantile(0.75),
                "q95": _values.quantile(0.95), "q99": _values.quantile(0.99),
                "min": _values.min(), "max": _values.max(), "skew": _values.skew(),
            })
    print(f"\n{_modality} feature summary by label (original units; full sample):")
    print(pd.DataFrame(_rows).round(4).to_string(index=False))

# News describes article-level corpus coverage and frozen article embeddings, not the model's
# past-only pooled news tensor. Keep the two units of analysis explicit.
_sample_start, _sample_end = _sample["Date"].min(), _sample["Date"].max()
if "day" in news.columns and len(news):
    _news = news.copy()
    _news["day"] = pd.to_datetime(_news["day"], errors="coerce")
    _news["src"] = _news["src"].fillna("unknown")
    _news = _news.dropna(subset=["day"])
    _news_window = _news.loc[_news["day"].between(_sample_start, _sample_end)].copy()
    _sources = sorted(_news["src"].unique().tolist())
    _fig, _axes = plt.subplots(3, 2, figsize=(15, 12), constrained_layout=True)
    _na = _axes.ravel()

    _corpus_counts = _news["src"].value_counts()
    _window_counts = _news_window["src"].value_counts()
    _x = np.arange(len(_sources))
    _width = 0.38
    _b1 = _na[0].bar(_x - _width / 2, [_corpus_counts.get(s, 0) for s in _sources], _width,
                     color="#718096", label="Full corpus")
    _b2 = _na[0].bar(_x + _width / 2, [_window_counts.get(s, 0) for s in _sources], _width,
                     color="#2878B5", label="Sample-date window")
    _na[0].set_xticks(_x, _sources, rotation=15, ha="right")
    _na[0].set_ylabel("Articles")
    _na[0].set_title("A. Article volume by source", loc="left", fontweight="bold")
    _na[0].legend(frameon=False, fontsize=8)
    _na[0].bar_label(_b1, fmt="%.0f", padding=2, fontsize=7)
    _na[0].bar_label(_b2, fmt="%.0f", padding=2, fontsize=7)
    _na[0].grid(axis="y", alpha=0.2)

    if len(_news_window):
        _monthly = _news_window.groupby([pd.Grouper(key="day", freq="MS"), "src"]).size().unstack(fill_value=0)
        for _src in _sources:
            if _src in _monthly:
                _na[1].plot(_monthly.index, _monthly[_src], marker="o", markersize=2.5,
                            linewidth=1.2, label=_src)
    _na[1].set_title("B. Monthly article volume", loc="left", fontweight="bold")
    _na[1].set_ylabel("Articles per month")
    _na[1].legend(frameon=False, fontsize=8)
    _na[1].grid(axis="y", alpha=0.2)

    _sample_days = pd.DatetimeIndex(_sample["Date"].drop_duplicates().sort_values())
    _daily_total = _news_window.groupby("day").size().reindex(_sample_days, fill_value=0)
    _daily_bins = np.arange(-0.5, max(1, int(_daily_total.max())) + 1.5, 1)
    _na[2].hist(_daily_total, bins=_daily_bins, color="#6B7FD7", edgecolor="white")
    _na[2].axvline(_daily_total.mean(), color="#D95F5F", linestyle="--",
                   label=f"Mean={_daily_total.mean():.2f}")
    _na[2].set_title("C. Articles per sample date (zeros included)", loc="left", fontweight="bold")
    _na[2].set_xlabel("Articles on sample date")
    _na[2].set_ylabel("Number of sample dates")
    _na[2].legend(frameon=False, fontsize=8)
    _na[2].grid(axis="y", alpha=0.2)

    _news_window["word_count"] = _news_window["text"].fillna("").str.split().str.len()
    _word_cap = max(1, float(_news_window["word_count"].quantile(0.99))) if len(_news_window) else 1
    for _src in _sources:
        _words = _news_window.loc[_news_window["src"] == _src, "word_count"]
        _words = _words[_words <= _word_cap]
        if len(_words):
            _na[3].hist(_words, bins=35, density=True, histtype="step", linewidth=1.6, label=_src)
    _na[3].set_title("D. Article text length by source", loc="left", fontweight="bold")
    _na[3].set_xlabel("Whitespace-delimited words (P99 display cap)")
    _na[3].set_ylabel("Density")
    _na[3].legend(frameon=False, fontsize=8)
    _na[3].grid(axis="y", alpha=0.2)

    _embedded = _news_window.loc[_news_window["vec"].notna()] if "vec" in _news_window else _news_window.iloc[0:0]
    if len(_embedded):
        _emb = np.stack(_embedded["vec"].to_numpy()).astype(float)
        _norm = np.linalg.norm(_emb, axis=1)
        for _src in _sources:
            _mask = _embedded["src"].to_numpy() == _src
            if _mask.any():
                _na[4].hist(_norm[_mask], bins=30, density=True, histtype="step", linewidth=1.6,
                            label=f"{_src} (n={_mask.sum():,})")
        _na[4].set_title("E. Frozen CLS embedding L2 norm", loc="left", fontweight="bold")
        _na[4].set_xlabel("L2 norm per article vector")
        _na[4].set_ylabel("Density")
        _na[4].legend(frameon=False, fontsize=7)
        _na[4].grid(axis="y", alpha=0.2)
        _coord = _emb.ravel()
        if len(_coord) > 250_000:
            _coord = _coord[np.random.default_rng(CFG.seed).choice(len(_coord), 250_000, replace=False)]
        _na[5].hist(_coord, bins=60, density=True, color="#718096", alpha=0.85)
        _na[5].set_title("F. CLS embedding coordinate values", loc="left", fontweight="bold")
        _na[5].set_xlabel("Coordinate value (random sample if >250k values)")
        _na[5].set_ylabel("Density")
        _na[5].grid(axis="y", alpha=0.2)
        print("\nNews CLS embedding summary (article-level vectors within sample-date window):")
        print(f"  vectors={len(_emb):,} | dimensions={_emb.shape[1]} | mean L2 norm={_norm.mean():.4f} | SD L2 norm={_norm.std():.4f}")
        print(f"  coordinate mean={_emb.mean():.5f} | coordinate SD={_emb.std():.5f} | mean per-dimension SD={_emb.std(axis=0).mean():.5f}")
    else:
        for _ax in _na[4:]:
            _ax.text(0.5, 0.5, "No article embeddings in this sample-date window.",
                     ha="center", va="center", wrap=True)
            _ax.set_title("Embedding diagnostic unavailable")
            _ax.set_xticks([])
            _ax.set_yticks([])

    _fig.suptitle("News corpus and frozen IndoBERT embedding distribution", fontsize=14, fontweight="bold")
    plt.show()

    _source_summary = _news.groupby("src").agg(
        corpus_articles=("text", "size"), first_day=("day", "min"), last_day=("day", "max"),
        corpus_active_days=("day", "nunique"),
        median_words=("text", lambda s: s.fillna("").str.split().str.len().median()),
    )
    _source_summary["sample_articles"] = _news_window.groupby("src").size()
    _source_summary["sample_active_days"] = _news_window.groupby("src")["day"].nunique()
    _source_summary[["sample_articles", "sample_active_days"]] = (
        _source_summary[["sample_articles", "sample_active_days"]].fillna(0).astype(int)
    )
    _source_summary["sample_date_coverage"] = _source_summary["sample_active_days"] / len(_sample_days)
    _source_summary["first_day"] = _source_summary["first_day"].dt.strftime("%Y-%m-%d")
    _source_summary["last_day"] = _source_summary["last_day"].dt.strftime("%Y-%m-%d")
    print("\nNews source coverage and text summary (corpus counts vs model date window):")
    print(_source_summary.round(3).to_string())
else:
    print("\nNews corpus distribution unavailable: no dated news articles are loaded.")

print("\nInterpretation: technical/macro rows are aligned to model sample dates; macro values can repeat due to point-in-time forward-fill. News plots use article-level records/CLS vectors, not model-pooled news tensors. This descriptive audit does not alter train-only scaling, labels, or model inputs.")

## Main Evaluation

Walk-forward: full model + 5 ablations (5 expanding-window splits).

In [ ]:
# --- Main walk-forward evaluation (thesis 2.6: full + 5 ablations) ---
VARIANTS = ("lstm", "lstm_macro", "lstm_news", "lstm_news_shuffled", "static", "full")
all_res = walk_forward_evaluate(df, CFG, n_splits=5, variants=VARIANTS, return_details=True)
print(f"{'variant':>18} | {'AUC':>6} | {'Acc@.5':>6} | {'F1@.5':>6} | F1_d/F1_u | Sharpe(strat/bh) | MDD(strat) | WinRate")
for v in VARIANTS:
    r, t = all_res[v], all_res[v]["trading"]["strategy"]
    bh = all_res[v]["trading"]["buyhold"]
    print(f"{v:>18} | {r['auc']:6.4f} | {r['acc']:6.4f} | {r['f1']:6.4f} | {r['f1_down']:.3f}/{r['f1_up']:.3f} | "
          f"{t['sharpe']:6.3f}/{bh['sharpe']:6.3f} | {t['max_drawdown']:.4f} | {t['win_rate']:.4f} / {bh['win_rate']:.4f}")
print("* Acc/F1 pooled at 0.5; fold-level Acc/F1 use train-selected Youden-J (AUC threshold-free)")
np.savez_compressed(ROOT / "database" / "wf_artifacts.npz", **{f"{v}_{k}": np.array(all_res[v][k])
    for v in VARIANTS for k in ("preds", "truths", "test_dates", "weights_list", "fold_indices")},
    horizon=np.array([CFG.horizon]), seed=np.array([CFG.seed]))
print("saved database/wf_artifacts.npz (predictions, labels, dates, fold IDs, and fold-mean model weights per variant)")

### Model Variant and Gate-Weight Table

Generated from the current `all_res` walk-forward training run. Adaptive gate weights are averaged across the test observations within each fold; the summary reports the mean and SD of those five fold means. `Gate status` explains each row: learned (adaptive), fixed (static), or no gate. An em dash means a modality weight is not defined for that architecture—not a missing measurement and not a zero weight. In this implementation, the shuffled-news control is the ungated LSTM+News model trained with shuffled news inputs; test inputs are unshuffled.

In [ ]:
# --- Detailed model-variant and modality-weight tables from this training run ---
from IPython.display import display

_variant_table_specs = {
    "full": {
        "Variant ID": "M1 (Full)",
        "Architecture configuration": "LSTM(7→64) + MLP(3→32→16) + news projection(768→64)",
        "Included input modalities": "Technical + macro + IndoBERT news",
        "Fusion mechanism": "Learned 3-way softmax gate",
        "gate_type": "adaptive",
    },
    "lstm": {
        "Variant ID": "M2 (Technical baseline)",
        "Architecture configuration": "LSTM(7→64) + binary head",
        "Included input modalities": "Technical only",
        "Fusion mechanism": "No fusion; single-modality head",
        "gate_type": "none",
    },
    "lstm_macro": {
        "Variant ID": "M3 (Technical + Macro)",
        "Architecture configuration": "LSTM(7→64) + MLP(3→32→16) + concatenated binary head",
        "Included input modalities": "Technical + macro",
        "Fusion mechanism": "Direct concatenation; no gating",
        "gate_type": "none",
    },
    "lstm_news": {
        "Variant ID": "M4 (Technical + News)",
        "Architecture configuration": "LSTM(7→64) + news projection(768→64) + concatenated binary head",
        "Included input modalities": "Technical + IndoBERT news",
        "Fusion mechanism": "Direct concatenation; no gating",
        "gate_type": "none",
    },
    "static": {
        "Variant ID": "M5 (Static multimodal)",
        "Architecture configuration": "LSTM(7→64) + MLP(3→32→16) + news projection(768→64)",
        "Included input modalities": "Technical + macro + IndoBERT news",
        "Fusion mechanism": "Fixed equal weights (1/3 each); not learned",
        "gate_type": "fixed",
    },
    "lstm_news_shuffled": {
        "Variant ID": "M6 (Shuffled-news control)",
        "Architecture configuration": "LSTM(7→64) + news projection(768→64) + concatenated binary head; training news shuffled within each fold",
        "Included input modalities": "Technical + shuffled IndoBERT news at training; original news at test",
        "Fusion mechanism": "Direct concatenation; no gating",
        "gate_type": "none",
    },
}
_table_variant_order = [v for v in ("full", "lstm", "lstm_macro", "lstm_news", "static", "lstm_news_shuffled") if v in VARIANTS]
_missing_results = [v for v in _table_variant_order if v not in all_res]
if _missing_results:
    raise KeyError(f"Run the main evaluation for all table variants first; missing: {_missing_results}")

_variant_summary_rows, _variant_fold_rows = [], []
for _v in _table_variant_order:
    _spec = _variant_table_specs[_v]
    _result = all_res[_v]
    _fold_weights = np.asarray(_result["weights_list"], dtype=float)
    _fold_meta = _result["fold_metrics"]
    if _fold_weights.shape != (len(_fold_meta), 3):
        raise ValueError(
            f"{_v}: expected one three-modality mean weight per fold; got "
            f"{_fold_weights.shape} for {len(_fold_meta)} folds. Re-run Main Evaluation."
        )
    if _spec["gate_type"] == "adaptive" and (
        not np.isfinite(_fold_weights).all()
        or np.any(_fold_weights < -1e-6)
        or not np.allclose(_fold_weights.sum(axis=1), 1.0, atol=1e-4)
    ):
        raise ValueError(f"{_v}: adaptive gate weights must be finite, nonnegative, and sum to one per fold.")
    if _spec["gate_type"] == "fixed" and not np.allclose(_fold_weights, 1 / 3, atol=1e-6):
        raise ValueError("Static multimodal variant should report the fixed 1/3, 1/3, 1/3 weights.")

    _gate_cols = ("Mean α (Technical)", "Mean β (Macro)", "Mean γ (News)")
    if _spec["gate_type"] == "adaptive":
        _means = _fold_weights.mean(axis=0)
        _sds = _fold_weights.std(axis=0, ddof=1) if len(_fold_weights) > 1 else np.zeros(3)
        _gate_summary = {col: f"{mean:.3f} ± {sd:.3f}" for col, mean, sd in zip(_gate_cols, _means, _sds)}
    elif _spec["gate_type"] == "fixed":
        _gate_summary = {col: "0.333 (fixed)" for col in _gate_cols}
    else:
        _gate_summary = {col: "—" for col in _gate_cols}
    _variant_summary_rows.append({
        "Variant ID": _spec["Variant ID"],
        "Architecture configuration": _spec["Architecture configuration"],
        "Included input modalities": _spec["Included input modalities"],
        "Fusion mechanism": _spec["Fusion mechanism"],
        "Gate status": {"adaptive": "Learned (adaptive)", "fixed": "Fixed (not learned)", "none": "No gate defined"}[_spec["gate_type"]],
        **_gate_summary,
    })
    for _i, (_fold, _weights) in enumerate(zip(_fold_meta, _fold_weights)):
        _row = {
            "Variant ID": _spec["Variant ID"],
            "Fold": int(_fold["fold"]),
            "Test n": int(_fold["n_samples"]),
            "Fold AUC": float(_fold["auc"]),
            "Gate status": {"adaptive": "Learned (adaptive)", "fixed": "Fixed (not learned)", "none": "No gate defined"}[_spec["gate_type"]],
        }
        if _spec["gate_type"] == "adaptive":
            _row.update(dict(zip(_gate_cols, [f"{w:.4f}" for w in _weights])))
        elif _spec["gate_type"] == "fixed":
            _row.update({col: "0.3333 (fixed)" for col in _gate_cols})
        else:
            _row.update({col: "—" for col in _gate_cols})
        _variant_fold_rows.append(_row)

_variant_gate_table = pd.DataFrame(_variant_summary_rows)
_variant_fold_table = pd.DataFrame(_variant_fold_rows)
display(_variant_gate_table.style.hide(axis="index").set_caption(
    "Model variants, gate status, and out-of-fold modality fusion weights"
))
display(_variant_fold_table.style.hide(axis="index").set_caption(
    "Fold-level detail: test-fold mean weights and corresponding fold AUC"
))
print(
    "Aggregation: each fold's test-sample gate weights are averaged first; the summary is "
    "the mean ± sample SD across folds. SD describes fold-to-fold variation, not a confidence interval. "
    "An em dash means no gate is defined for that architecture; fixed weights are not learned estimates."
)

### Walk-Forward Result Visualizations

Fold-level observations are shown separately from pooled metrics. Points are individual folds (not confidence intervals); pooled diamonds summarize the concatenated out-of-fold predictions. Fold Accuracy/F1 use train-selected Youden-J thresholds, while pooled Accuracy/F1 use the fixed 0.5 threshold shown in the table.

In [ ]:
# --- Scientist-oriented visualization of walk-forward results ---
import seaborn as sns
from matplotlib.lines import Line2D

sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")
_variant_order = list(VARIANTS)
_variant_names = {
    "lstm": "LSTM\n(technical)",
    "lstm_macro": "LSTM +\nmacro",
    "lstm_news": "LSTM +\nnews",
    "lstm_news_shuffled": "News\n(shuffled)",
    "static": "Static\n(multimodal)",
    "full": "Full\n(multimodal)",
}
_tick_names = [_variant_names.get(v, v.replace("_", " ")) for v in _variant_order]
_fold_rows, _pooled_rows, _trading_fold_rows, _trading_pooled_rows = [], [], [], []
for _variant in _variant_order:
    _result = all_res[_variant]
    _pooled_rows.append({
        "variant": _variant,
        "AUC": _result["auc"],
        "Accuracy": _result["acc"],
        "F1 (macro)": _result["f1"],
        "F1 (DOWN)": _result["f1_down"],
        "F1 (UP)": _result["f1_up"],
    })
    for _fold in _result["fold_metrics"]:
        _fold_rows.append({"variant": _variant, "fold": _fold["fold"], **{
            _name: _fold[_key] for _name, _key in (
                ("AUC", "auc"), ("Accuracy", "acc"), ("F1 (macro)", "f1"),
                ("F1 (DOWN)", "f1_down"), ("F1 (UP)", "f1_up"),
            )
        }})
    for _side in ("strategy", "buyhold"):
        _trade_summary = _result["trading"][_side]
        _trading_pooled_rows.append({
            "variant": _variant, "series": _side,
            "Sharpe ratio": _trade_summary["sharpe"],
            "Maximum drawdown": _trade_summary["max_drawdown"],
            "Win rate": _trade_summary["win_rate"],
        })
    for _fold in _result["fold_trading"]:
        for _side in ("strategy", "buyhold"):
            _trading_fold_rows.append({
                "variant": _variant, "fold": _fold["fold"], "series": _side,
                **{_name: _fold[_side][_key] for _name, _key in (
                    ("Sharpe ratio", "sharpe"),
                    ("Maximum drawdown", "max_drawdown"),
                    ("Win rate", "win_rate"),
                )},
            })

_pooled_perf = pd.DataFrame(_pooled_rows).set_index("variant").reindex(_variant_order)
_fold_perf = pd.DataFrame(_fold_rows)
_perf_metrics = ["AUC", "Accuracy", "F1 (macro)", "F1 (DOWN)", "F1 (UP)"]
_perf_fig, _perf_axes = plt.subplots(2, 3, figsize=(17, 9), sharey=True)
for _ax, _metric in zip(_perf_axes.flat, _perf_metrics):
    sns.stripplot(
        data=_fold_perf, x="variant", y=_metric, order=_variant_order,
        color="#4C78A8", jitter=0.16, size=5.5, alpha=0.78, ax=_ax,
    )
    _ax.scatter(
        np.arange(len(_variant_order)), _pooled_perf[_metric],
        marker="D", s=66, color="#C44E52", edgecolor="white", linewidth=0.7,
        zorder=5,
    )
    _ax.set_title(_metric, loc="left", fontweight="bold")
    _ax.set_xlabel("")
    _ax.set_ylabel("Score" if _ax in (_perf_axes[0, 0], _perf_axes[1, 0]) else "")
    _ax.set_ylim(-0.03, 1.03)
    _ax.set_xticks(np.arange(len(_variant_order)), _tick_names, rotation=0, fontsize=8)
    _ax.grid(axis="x", visible=False)
    if _metric == "AUC":
        _ax.axhline(0.5, color="#555555", linestyle="--", linewidth=1, alpha=0.8)
        _ax.text(0.99, 0.505, "chance", transform=_ax.get_yaxis_transform(),
                 ha="right", va="bottom", fontsize=8, color="#555555")
    elif _metric == "Accuracy":
        _truth = np.asarray(all_res[_variant_order[0]]["truths"], dtype=int)
        _majority = max(np.mean(_truth == 0), np.mean(_truth == 1))
        _ax.axhline(_majority, color="#777777", linestyle=":", linewidth=1, alpha=0.9)
        _ax.text(0.99, _majority + 0.008, f"majority baseline={_majority:.2f}",
                 transform=_ax.get_yaxis_transform(), ha="right", va="bottom",
                 fontsize=7, color="#555555")
_perf_axes[1, 2].set_visible(False)
_perf_handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#4C78A8",
           markeredgecolor="none", markersize=7, label="Individual fold"),
    Line2D([0], [0], marker="D", color="none", markerfacecolor="#C44E52",
           markeredgecolor="white", markersize=7, label="Pooled out-of-fold"),
]
_perf_fig.legend(handles=_perf_handles, loc="upper center", ncol=2,
                 bbox_to_anchor=(0.5, 1.01), frameon=False)
_perf_fig.suptitle("Predictive performance by model variant", fontsize=15, fontweight="bold", y=1.05)
_perf_fig.text(
    0.5, 0.005,
    "Fold dots show fold_metrics; diamonds reproduce pooled values above. AUC is threshold-free. "
    "Fold Accuracy/F1 use train-selected Youden-J thresholds; pooled Accuracy/F1 use fixed 0.5 threshold.",
    ha="center", fontsize=8, color="#555555",
)
_perf_fig.tight_layout(rect=[0, 0.04, 1, 0.94])
plt.show()

# Fold-wise strategy and buy-and-hold observations, with pooled summaries overlaid as diamonds.
_trade_folds = pd.DataFrame(_trading_fold_rows)
_trade_pooled = pd.DataFrame(_trading_pooled_rows)
_trade_metrics = ["Sharpe ratio", "Maximum drawdown", "Win rate"]
_series_order = ["strategy", "buyhold"]
_series_palette = {"strategy": "#2878B5", "buyhold": "#7A7A7A"}
_trade_fig, _trade_axes = plt.subplots(1, 3, figsize=(18, 6))
for _ax, _metric in zip(_trade_axes, _trade_metrics):
    sns.stripplot(
        data=_trade_folds, x="variant", y=_metric, hue="series",
        order=_variant_order, hue_order=_series_order, palette=_series_palette,
        dodge=True, jitter=0.12, size=5, alpha=0.72, ax=_ax,
    )
    for _side, _offset in (("strategy", -0.2), ("buyhold", 0.2)):
        _summary = _trade_pooled.loc[_trade_pooled["series"] == _side].set_index("variant").reindex(_variant_order)
        _ax.scatter(
            np.arange(len(_variant_order)) + _offset, _summary[_metric],
            marker="D", s=64, color=_series_palette[_side], edgecolor="white",
            linewidth=0.7, zorder=5,
        )
    _ax.set_title(_metric, loc="left", fontweight="bold")
    _ax.set_xlabel("")
    _ax.set_ylabel(_metric)
    _ax.set_xticks(np.arange(len(_variant_order)), _tick_names, rotation=0, fontsize=8)
    _ax.grid(axis="x", visible=False)
    if _metric in ("Sharpe ratio", "Maximum drawdown"):
        _ax.axhline(0, color="#555555", linestyle="--", linewidth=1, alpha=0.8)
    if _metric == "Win rate":
        _ax.axhline(0.5, color="#555555", linestyle="--", linewidth=1, alpha=0.8)
_trade_handles = [
    Line2D([0], [0], marker="o", linestyle="none", color=_series_palette[s],
           markersize=6, label=f"{s.title()} — fold") for s in _series_order
] + [
    Line2D([0], [0], marker="D", linestyle="none", color=_series_palette[s],
           markersize=6, label=f"{s.title()} — pooled") for s in _series_order
]
_trade_fig.legend(handles=_trade_handles, loc="upper center", ncol=4,
                  bbox_to_anchor=(0.5, 1.04), frameon=False)
_trade_fig.suptitle("Trading outcomes: strategy versus buy-and-hold", fontsize=15,
                    fontweight="bold", y=1.1)
_trade_fig.text(
    0.5, -0.02,
    "Circles are the five walk-forward folds; diamonds summarize concatenated out-of-fold dates. "
    "Fold variation is descriptive (not a confidence interval). Sharpe is annualized; drawdown is negative peak-to-trough loss.",
    ha="center", fontsize=8, color="#555555",
)
_trade_fig.tight_layout(rect=[0, 0.04, 1, 0.9])
plt.show()

print("Each point is a fold, not an independent experiment; these displays are descriptive and do not replace paired fold-level inference.")

## Cost Analysis & Per-Fold Trading

Transaction costs (0/15/25 bps) and fold-level trading metrics.

In [ ]:
# --- Cost analysis & per-fold trading (full variant) ---
for _c in (0.0, 15.0, 25.0):
    _cf = trading_stats(all_res["full"]["test_dates"], all_res["full"]["preds"], df, cost_bps=_c)["strategy"]
    _cb = trading_stats(all_res["full"]["test_dates"], all_res["full"]["preds"], df, cost_bps=_c)["buyhold"]
    print(f"cost {_c:4.1f} bps: Sharpe={_cf['sharpe']:.3f}/{_cb['sharpe']:.3f} MDD={_cf['max_drawdown']:.4f} n_trades={_cf['n_trades']}")

print("per-fold trading (full, thr=0.5): fold | Sharpe_strat | MDD_strat | win_strat")
for _fd in all_res["full"]["fold_trading"]:
    _t = _fd["strategy"]
    print(f"  {_fd['fold']} | {_t['sharpe']:6.3f} | {_t['max_drawdown']:.4f} | {_t['win_rate']:.4f}")

## Horizon Sensitivity (Full Model)

Rebuilds the forward-direction labels and refits the full model at t+1, t+5, and t+10. The primary t+5 result reuses the main walk-forward run. `n (samples)` is the total usable model-window count; `n (OOF)` is the subset evaluated by the five-fold walk-forward test sets. News gate γ is the mean out-of-fold news-modality soft-gate weight. ΔAUC is horizon minus t+5, paired by walk-forward fold (unadjusted 95% paired t interval; five folds). Strategy and buy-and-hold Sharpe use the notebook's unchanged five-trading-day holding convention at every horizon.

In [ ]:
# --- Horizon sensitivity: horizon-specific labels, purged folds, and fresh model fits ---
from IPython.display import display

_horizons = (1, 5, 10)
_horizon_results = {5: all_res["full"]}
_horizon_data = {5: df}
for _h in (1, 10):
    _df_h = align(*frames, news_daily, horizon=_h)
    _cfg_h = dataclasses.replace(CFG, horizon=_h)
    _horizon_data[_h] = _df_h
    _horizon_results[_h] = walk_forward_evaluate(
        _df_h, _cfg_h, n_splits=5, variants=("full",), return_details=True
    )["full"]

_reference_fold_auc = np.asarray(
    [fold["auc"] for fold in _horizon_results[5]["fold_metrics"]], dtype=float
)
_horizon_rows = []
for _h in _horizons:
    _result_h = _horizon_results[_h]
    _fold_auc_h = np.asarray([fold["auc"] for fold in _result_h["fold_metrics"]], dtype=float)
    if len(_fold_auc_h) != len(_reference_fold_auc):
        raise ValueError(f"Horizon t+{_h} has a different number of walk-forward folds.")
    if _h == 5:
        _delta_auc, _delta_ci_low, _delta_ci_high = 0.0, np.nan, np.nan
    else:
        _delta_auc, _delta_ci_low, _delta_ci_high = paired_stats(
            _fold_auc_h, _reference_fold_auc
        )
    _gate_weights = np.stack(_result_h["weights_list"])
    if _gate_weights.ndim != 2 or _gate_weights.shape[1] != 3:
        raise ValueError(f"Unexpected full-model gate-weight shape at t+{_h}: {_gate_weights.shape}")
    _strategy = _result_h["trading"]["strategy"]
    _buyhold = _result_h["trading"]["buyhold"]
    _horizon_rows.append({
        "Horizon": f"t+{_h}",
        "n (samples)": len(_horizon_data[_h]) - CFG.look_back + 1,
        "n (OOF)": len(_result_h["truths"]),
        "Pooled AUC": _result_h["auc"],
        "Fold AUC (mean ± SD)": f"{_fold_auc_h.mean():.3f} ± {_fold_auc_h.std(ddof=1):.3f}",
        "γ (news gate)": _gate_weights[:, 2].mean(),
        "Sharpe (strategy / buy-hold)": f"{_strategy['sharpe']:.3f} / {_buyhold['sharpe']:.3f}",
        "ΔAUC vs t+5 (95% CI)": (
            "Reference" if _h == 5 else
            f"{_delta_auc:+.3f} [{_delta_ci_low:+.3f}, {_delta_ci_high:+.3f}]"
        ),
    })

_horizon_table = pd.DataFrame(_horizon_rows)
display(_horizon_table.style.hide(axis="index").format({
    "Pooled AUC": "{:.4f}", "γ (news gate)": "{:.3f}"
}).set_caption(
    "Horizon sensitivity — Full adaptive soft-gating model"
))
print(
    "Inference note: paired intervals compare fold-level AUCs by split index, not identical "
    "observation dates; label horizon changes the evaluated samples. Intervals spanning zero "
    "do not show a clear difference and do not establish equivalence."
)
horizon_sensitivity = _horizon_results

## Statistical Inference

Bootstrap CI, classical baselines, paired ΔAUC tests, ablations — all read-only over stored outputs.

In [ ]:
# --- Statistical inference (read-only over stored outputs) ---
# Sharpe block-bootstrap CI
_sd = np.array(all_res["full"]["trading"]["strategy"]["daily"])
_rng = np.random.default_rng(CFG.seed)
_blk, _n = 5, len(_sd)
_nb = int(np.ceil(_n / _blk))
_bs = []
for _ in range(2000):
    _idx = np.concatenate([np.arange(_s, min(_s + _blk, _n)) for _s in _rng.integers(0, _n, _nb)])[:_n]
    _x = _sd[_idx]
    _bs.append(float(_x.mean() / (_x.std() + 1e-9) * np.sqrt(252)))
print(f"Sharpe 95% block-bootstrap CI (full, 2000 resamples, block=5): [{np.percentile(_bs, 2.5):.3f}, {np.percentile(_bs, 97.5):.3f}]")

# Classical baselines + paired vs full
for _m in ("logreg", "rf"):
    _r = classical_baseline(df, CFG, model=_m)
    _t, _bh = _r["trading"]["strategy"], _r["trading"]["buyhold"]
    print(f"classical_{_m:>6} | {_r['auc']:6.4f} | {_r['acc']:6.4f} | {_r['f1']:6.4f} | {_r['f1_down']:.3f}/{_r['f1_up']:.3f} | "
          f"{_t['sharpe']:6.3f}/{_bh['sharpe']:6.3f} | {_t['max_drawdown']:.4f} | {_t['win_rate']:.4f} / {_bh['win_rate']:.4f}")
    _fa = [f["auc"] for f in all_res["full"]["fold_metrics"]]
    _ca = [f["auc"] for f in _r["fold_metrics"]]
    _d, _lo, _hi = paired_stats(_ca, _fa)
    print(f"  paired ΔAUC classical_{_m}-vs-full: {_d:+.4f} [{_lo:+.4f}, {_hi:+.4f}]" + ("  ~noise" if _lo < 0 < _hi else ""))

# Paired stats: full vs ablations (per-fold AUC)
f_auc = [m["auc"] for m in all_res["full"]["fold_metrics"]]
print("paired ΔAUC vs full [95% CI]:")
for v in VARIANTS:
    if v in ("full", "lstm_news_shuffled"):
        continue
    d, lo, hi = paired_stats(f_auc, [m["auc"] for m in all_res[v]["fold_metrics"]])
    print(f"  full-{v}: {d:+.4f} [{lo:+.4f}, {hi:+.4f}]" + ("  ~noise" if lo < 0 < hi else ""))
_d, _lo, _hi = paired_stats([m["auc"] for m in all_res["lstm_macro"]["fold_metrics"]], [m["auc"] for m in all_res["lstm"]["fold_metrics"]])
print(f"RQ1 macro-vs-lstm dAUC: {_d:+.4f} [{_lo:+.4f}, {_hi:+.4f}]" + ("  ~noise" if _lo < 0 < _hi else ""))

# Focal loss vs BCE (full)
focal_res = walk_forward_evaluate(df, CFG, n_splits=5, variants=("full",), loss="focal")
fr = focal_res["full"]
print(f"(2) focal | {fr['auc']:6.4f} | {fr['acc']:6.4f} | {fr['f1']:6.4f} | (vs bce {all_res['full']['auc']:.4f}/{all_res['full']['acc']:.4f}/{all_res['full']['f1']:.4f})")

# Gradient audit (epoch 0)
g = all_res["full"]["grad_norms"]
print("(4) grad0 " + " ".join(f"{k}={np.mean([x[k] for x in g]):.3f}" for k in ("lstm", "mlp", "news", "gate+head")))

# News aggregation ablation (last vs mean)
last_res = walk_forward_evaluate(df, CFG, n_splits=5, variants=("full",), news_agg="last")
lr = last_res["full"]
print(f"E3 news_agg=last | {lr['auc']:6.4f} | {lr['acc']:6.4f} | {lr['f1']:6.4f} | (vs mean {all_res['full']['auc']:.4f}/{all_res['full']['acc']:.4f}/{all_res['full']['f1']:.4f})")

# Learning curves (full)
print("E2 learning curves (full): fold | epochs | train_loss | val_loss")
for i, h in enumerate(all_res["full"]["learning_curves"]):
    print(f"  fold {i} | {h['epochs_run']:>3} | {h['train'][-1]:.4f} | {h['val'][-1]:.4f}")

# Lookback x LR grid (full)
print("(5) grid lb x lr: AUC pooled")
grid = {}
for lb_ in (14, 28, 56):
    row = []
    for lr_ in (1e-3, 3e-4):
        gr = walk_forward_evaluate(df, CFG, n_splits=5, variants=("full",), lr=lr_, look_back=lb_)
        grid[(lb_, lr_)] = gr["full"]["auc"]
        row.append(f"{gr['full']['auc']:.4f}")
    print(f"  lb={lb_:>2}: " + "  ".join(f"lr={l}: {a}" for l, a in zip(("1e-3", "3e-4"), row)))

# Store for downstream viz
wf_metrics = all_res["full"]
wf_metrics["news_agg_last"] = {"auc": lr["auc"], "acc": lr["acc"], "f1": lr["f1"]}
wf_metrics["focal"] = {"auc": fr["auc"], "acc": fr["acc"], "f1": fr["f1"]}
wf_metrics["grid"] = {f"{k[0]}/{k[1]}": v for k, v in grid.items()}
wf_metrics["ablation"] = {v: {"auc": all_res[v]["auc"], "acc": all_res[v]["acc"], "f1": all_res[v]["f1"], "trading": all_res[v]["trading"], "preds": all_res[v]["preds"], "truths": all_res[v]["truths"], "fold_metrics": all_res[v]["fold_metrics"]} for v in VARIANTS}
model = None

## Visualize

Price + trend/momentum, macro alignment, label balance.

In [ ]:
# --- Technical Indicator Diagnostic Panel (FULL Data Range) ---
plt.style.use("seaborn-v0_8-whitegrid")
colors = {"close": "#1f1f1f", "sma": "#2E86C1", "macd": "#2E86C1", "signal": "#E74C3C",
          "hist_pos": "#27AE60", "hist_neg": "#E74C3C", "rsi": "#8E44AD",
          "overbought": "#E74C3C", "oversold": "#27AE60", "neutral": "#95A5A6",
          "atr": "#F39C12", "vol": "#7F8C8D"}

fig = plt.figure(figsize=(16, 11))
gs = fig.add_gridspec(5, 1, height_ratios=[2.5, 1.2, 1, 1, 1], hspace=0.08, wspace=0)
fig.subplots_adjust(top=0.92, bottom=0.10, left=0.09, right=0.98, hspace=0.22)

# 1) Price + SMA20 + Bands
ax1 = fig.add_subplot(gs[0])
ax1.plot(df["Date"], df["Close"], label="JCI Close", color=colors["close"], linewidth=0.8, alpha=0.85)
ax1.plot(df["Date"], df["SMA_20"], label="SMA(20)", color=colors["sma"], linewidth=0.9, alpha=0.85)
sma_std = df["Close"].rolling(20).std()
ax1.fill_between(df["Date"], df["SMA_20"] - 2*sma_std, df["SMA_20"] + 2*sma_std, color=colors["sma"], alpha=0.08, label="±2σ Bands")
ax1.set_ylabel("Price (IDR)", fontsize=10, fontweight="medium")
ax1.set_title("JCI Technical Indicators — Full Historical Range", fontsize=13, fontweight="bold", pad=12)
ax1.legend(fontsize=9, loc="upper left", ncol=3, framealpha=0.9)
ax1.grid(True, alpha=0.25, linestyle="-", linewidth=0.5)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

last_close = df["Close"].iloc[-1]
last_sma = df["SMA_20"].iloc[-1]
regime = "BULLISH" if last_close > last_sma else "BEARISH"
regime_color = colors["hist_pos"] if regime == "BULLISH" else colors["hist_neg"]
ax1.annotate(f"Current: {regime} (Close={last_close:,.0f} vs SMA20={last_sma:,.0f})",
             xy=(0.98, 0.96), xycoords="axes fraction", fontsize=9, fontweight="bold", color=regime_color,
             ha="right", va="top", bbox={"boxstyle": "round,pad=0.4", "facecolor": "white", "alpha": 0.9, "edgecolor": regime_color})

# 2) MACD
ax2 = fig.add_subplot(gs[1], sharex=ax1)
macd, signal = df["MACD"], df["MACD_Signal"]
hist = macd - signal
ax2.plot(df["Date"], macd, label="MACD(12,26)", color=colors["macd"], linewidth=0.8)
ax2.plot(df["Date"], signal, label="Signal(9)", color=colors["signal"], linewidth=0.8, alpha=0.9)
ax2.axhline(0, color="gray", linewidth=0.6, linestyle="-", alpha=0.5)
ax2.bar(df["Date"][hist >= 0], hist[hist >= 0], width=2.5, color=colors["hist_pos"], alpha=0.6, label="Histogram (+)")
ax2.bar(df["Date"][hist < 0], hist[hist < 0], width=2.5, color=colors["hist_neg"], alpha=0.6, label="Histogram (-)")
ax2.set_ylabel("MACD", fontsize=10, fontweight="medium")
ax2.legend(fontsize=8, loc="upper left", ncol=4, framealpha=0.9)
ax2.grid(True, alpha=0.25, linestyle="-", linewidth=0.5)

# 3) RSI
ax3 = fig.add_subplot(gs[2], sharex=ax1)
ax3.plot(df["Date"], df["RSI_14"], label="RSI(14)", color=colors["rsi"], linewidth=0.8)
ax3.axhspan(70, 100, color=colors["overbought"], alpha=0.12, label="Overbought (>70)")
ax3.axhspan(30, 70, color=colors["neutral"], alpha=0.06, label="Neutral")
ax3.axhspan(0, 30, color=colors["oversold"], alpha=0.12, label="Oversold (<30)")
ax3.axhline(70, color=colors["overbought"], linewidth=0.9, linestyle="--", alpha=0.8)
ax3.axhline(50, color=colors["neutral"], linewidth=0.7, linestyle=":", alpha=0.7)
ax3.axhline(30, color=colors["oversold"], linewidth=0.9, linestyle="--", alpha=0.8)
ax3.set_ylabel("RSI", fontsize=10, fontweight="medium")
ax3.set_ylim(0, 100)
ax3.set_yticks([0, 30, 50, 70, 100])
ax3.legend(fontsize=8, loc="upper left", ncol=4, framealpha=0.9)
ax3.grid(True, alpha=0.25, linestyle="-", linewidth=0.5)

# 4) ATR
ax4 = fig.add_subplot(gs[3], sharex=ax1)
ax4.plot(df["Date"], df["ATR_14"], label="ATR(14)", color=colors["atr"], linewidth=0.8)
atr_pct = df["ATR_14"].rolling(120, min_periods=40).rank(pct=True)
high_vol = atr_pct > 0.8
if high_vol.any():
    ax4.scatter(df["Date"][high_vol], df["ATR_14"][high_vol], c=colors["atr"], s=10, alpha=0.5, zorder=5, label="High Vol (>80th pct)")
ax4.set_ylabel("ATR (IDR)", fontsize=10, fontweight="medium")
ax4.legend(fontsize=8, loc="upper left", framealpha=0.9)
ax4.grid(True, alpha=0.25, linestyle="-", linewidth=0.5)
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# 5) Volume
ax5 = fig.add_subplot(gs[4], sharex=ax1)
vol_colors = ["#27AE60" if df["Close"].iloc[i] >= df["Close"].iloc[i-1] else "#E74C3C" for i in range(1, len(df))]
ax5.bar(df["Date"].iloc[1:], df["Volume"].iloc[1:], width=2.5, color=vol_colors, alpha=0.45, label="Volume")
ax5.plot(df["Date"], df["Volume"].rolling(60).mean(), color=colors["close"], linewidth=1.2, label="Vol SMA(60)")
ax5.set_ylabel("Volume", fontsize=10, fontweight="medium")
ax5.set_xlabel("Date", fontsize=10, fontweight="medium")
ax5.legend(fontsize=8, loc="upper left", framealpha=0.9)
ax5.grid(True, alpha=0.25, linestyle="-", linewidth=0.5)
ax5.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))

import matplotlib.dates as mdates

ax5.xaxis.set_major_locator(mdates.YearLocator())
ax5.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
for ax in (ax1, ax2, ax3, ax4):
    ax.tick_params(axis="x", labelbottom=False)
plt.show()

tech_cols_ext = CFG.tech_cols + ["ATR_14", "RSI_14", "MACD"]
tech_summary = df[tech_cols_ext].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
print("Technical Feature Summary (Full Sample):")
print(tech_summary.round(2).to_string())

In [ ]:
# --- Macro Data Diagnostic Panel (Point-in-Time Ffill Alignment) ---
plt.style.use("seaborn-v0_8-whitegrid")
mcolors = {"bi": "#2E86C1", "inf": "#27AE60", "fx": "#E67E22", "grid": "#BDC3C7"}

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# 1) BI Rate
ax = axes[0]
ax.step(df["Date"], df["bi_rate"], where="post", label="BI 7-Day Reverse Repo Rate (%)", color=mcolors["bi"], linewidth=1.1, alpha=0.9)
bi_diff = df["bi_rate"].diff()
if (bi_diff > 0).any():
    ax.scatter(df["Date"][bi_diff > 0], df["bi_rate"][bi_diff > 0], color=mcolors["bi"], s=35, marker="^", zorder=5, label="Rate Hike", alpha=0.8)
if (bi_diff < 0).any():
    ax.scatter(df["Date"][bi_diff < 0], df["bi_rate"][bi_diff < 0], color=mcolors["bi"], s=35, marker="v", zorder=5, label="Rate Cut", alpha=0.8)
curr_bi = df["bi_rate"].iloc[-1]
ax.axhline(curr_bi, color=mcolors["bi"], linewidth=0.8, linestyle=":", alpha=0.5)
ax.annotate(f"Current: {curr_bi:.2f}%", xy=(0.98, 0.95), xycoords="axes fraction", fontsize=9, fontweight="bold", color=mcolors["bi"], ha="right", va="top", bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.9, "edgecolor": mcolors["bi"]})
ax.set_ylabel("BI Rate (%)", fontsize=10, fontweight="medium")
ax.set_title("Macroeconomic Data — Point-in-Time Alignment (Forward-Fill)", fontsize=13, fontweight="bold", pad=12)
ax.legend(fontsize=9, loc="upper left", framealpha=0.9)
ax.grid(True, alpha=0.25, linestyle="-", linewidth=0.5)

# 2) Inflation
ax = axes[1]
ax.step(df["Date"], df["inflation_rate"], where="post", label="Inflation Rate (YoY %)", color=mcolors["inf"], linewidth=1.1, alpha=0.9)
ax.axhspan(2, 4, color=mcolors["inf"], alpha=0.08, label="BI Target Band (2-4%)")
curr_inf = df["inflation_rate"].iloc[-1]
ax.axhline(curr_inf, color=mcolors["inf"], linewidth=0.8, linestyle=":", alpha=0.5)
ax.annotate(f"Current: {curr_inf:.2f}%", xy=(0.98, 0.95), xycoords="axes fraction", fontsize=9, fontweight="bold", color=mcolors["inf"], ha="right", va="top", bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.9, "edgecolor": mcolors["inf"]})
ax.set_ylabel("Inflation (YoY %)", fontsize=10, fontweight="medium")
ax.legend(fontsize=9, loc="upper left", framealpha=0.9)
ax.grid(True, alpha=0.25, linestyle="-", linewidth=0.5)

# 3) USD/IDR
ax = axes[2]
ax.plot(df["Date"], df["usd_idr"], label="USD/IDR Spot Rate", color=mcolors["fx"], linewidth=0.9, alpha=0.85)
ax.plot(df["Date"], df["usd_idr"].rolling(60).mean(), label="SMA(60)", color=mcolors["grid"], linewidth=1.2, alpha=0.8)
curr_fx = df["usd_idr"].iloc[-1]
ax.axhline(curr_fx, color=mcolors["fx"], linewidth=0.8, linestyle=":", alpha=0.5)
ax.annotate(f"Current: {curr_fx:,.0f}", xy=(0.98, 0.95), xycoords="axes fraction", fontsize=9, fontweight="bold", color=mcolors["fx"], ha="right", va="top", bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.9, "edgecolor": mcolors["fx"]})
ax.set_ylabel("USD/IDR", fontsize=10, fontweight="medium")
ax.set_xlabel("Date", fontsize=10, fontweight="medium")
ax.legend(fontsize=9, loc="upper left", framealpha=0.9)
ax.grid(True, alpha=0.25, linestyle="-", linewidth=0.5)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

fig.autofmt_xdate(rotation=25, ha="right")
plt.tight_layout()
plt.show()

macro_summary = df[CFG.macro_cols].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
print("Macro Feature Summary (Full Sample):")
print(macro_summary.round(4).to_string())

# Label Balance
counts = df["target_direction"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(["DOWN (0)", "UP (1)"], [counts.get(0.0, 0), counts.get(1.0, 0)], color=["#E74C3C", "#27AE60"], alpha=0.75, edgecolor="white", linewidth=1.5, width=0.6)
for bar, val in zip(bars, [counts.get(0.0, 0), counts.get(1.0, 0)]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + max(counts)*0.02, f"{int(val):,} ({val/len(df):.1%})", ha='center', va='bottom', fontsize=10, fontweight="bold")
ax.set_ylabel("Count", fontsize=11, fontweight="medium")
ax.set_title(f"t+{CFG.horizon} Label Balance (n={len(df):,})", fontsize=12, fontweight="bold", pad=10)
ax.grid(True, alpha=0.25, axis='y')
ax.set_ylim(0, max(counts) * 1.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

imbalance_ratio = max(counts) / min(counts)
dominant = "UP" if counts.get(1.0, 0) > counts.get(0.0, 0) else "DOWN"
print(f"Class Imbalance Ratio: {imbalance_ratio:.2f}:1 ({dominant} dominant)")

## Feature Analysis

Correlation matrices and class-conditional distributions for feature engineering insights.

In [ ]:
# --- Feature Correlation & Distribution Analysis ---
# For scientist/engineer: understand feature relationships and distributions
plt.style.use("seaborn-v0_8-whitegrid")

# 1) Correlation Heatmap (Technical + Macro)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Technical features correlation
tech_corr = df[CFG.tech_cols + ["ATR_14", "RSI_14", "MACD"]].corr()
im1 = axes[0].imshow(tech_corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
axes[0].set_xticks(range(len(tech_corr.columns)))
axes[0].set_yticks(range(len(tech_corr.columns)))
axes[0].set_xticklabels(tech_corr.columns, rotation=45, ha="right", fontsize=9)
axes[0].set_yticklabels(tech_corr.columns, fontsize=9)
axes[0].set_title("Technical Features Correlation", fontsize=12, fontweight="bold", pad=10)
for i in range(len(tech_corr)):
    for j in range(len(tech_corr)):
        val = tech_corr.iloc[i,j]
        text_color = "white" if abs(val) > 0.5 else "black"
        axes[0].text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=text_color)
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

# Macro features correlation
macro_corr = df[CFG.macro_cols].corr()
im2 = axes[1].imshow(macro_corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
axes[1].set_xticks(range(len(macro_corr.columns)))
axes[1].set_yticks(range(len(macro_corr.columns)))
axes[1].set_xticklabels(macro_corr.columns, rotation=45, ha="right", fontsize=9)
axes[1].set_yticklabels(macro_corr.columns, fontsize=9)
axes[1].set_title("Macro Features Correlation", fontsize=12, fontweight="bold", pad=10)
for i in range(len(macro_corr)):
    for j in range(len(macro_corr)):
        val = macro_corr.iloc[i,j]
        text_color = "white" if abs(val) > 0.5 else "black"
        axes[1].text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=text_color)
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# 2) Feature Distributions (Violin plots for Technical)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
tech_feats = CFG.tech_cols + ["ATR_14", "RSI_14"]
for idx, feat in enumerate(tech_feats[:8]):
    if idx >= len(axes):
        break
    # Split by target direction
    down_data = df[df["target_direction"] == 0][feat].dropna()
    up_data = df[df["target_direction"] == 1][feat].dropna()
    parts = axes[idx].violinplot([down_data, up_data], positions=[0, 1], showmeans=True, showmedians=True)
    for pc, color in zip(parts['bodies'], ["#E74C3C", "#27AE60"]):
        pc.set_facecolor(color)
        pc.set_alpha(0.6)
    axes[idx].set_title(feat, fontsize=10, fontweight="medium")
    axes[idx].set_xticks([0, 1])
    axes[idx].set_xticklabels(["DOWN", "UP"])
    axes[idx].grid(True, alpha=0.25, axis='y')
    if idx == 0:
        axes[idx].set_ylabel("Value", fontsize=9)

plt.suptitle("Feature Distributions by Target Class (Technical)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# 3) Macro distributions by target
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for idx, feat in enumerate(CFG.macro_cols):
    down_data = df[df["target_direction"] == 0][feat].dropna()
    up_data = df[df["target_direction"] == 1][feat].dropna()
    parts = axes[idx].violinplot([down_data, up_data], positions=[0, 1], showmeans=True, showmedians=True)
    for pc, color in zip(parts['bodies'], ["#E74C3C", "#27AE60"]):
        pc.set_facecolor(color)
        pc.set_alpha(0.6)
    axes[idx].set_title(feat.replace("_", " ").title(), fontsize=11, fontweight="medium")
    axes[idx].set_xticks([0, 1])
    axes[idx].set_xticklabels(["DOWN", "UP"])
    axes[idx].grid(True, alpha=0.25, axis='y')
    if idx == 0:
        axes[idx].set_ylabel("Value", fontsize=10)

plt.suptitle("Feature Distributions by Target Class (Macro)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# 4) Print key correlations with target
print("Key Feature Correlations with Target (t+5 Direction):")
all_feats = CFG.tech_cols + ["ATR_14", "RSI_14", "MACD"] + CFG.macro_cols
target_corrs = df[all_feats + ["target_direction"]].corr()["target_direction"].drop("target_direction").sort_values(key=abs, ascending=False)
for feat, corr in target_corrs.head(15).items():
    print(f"  {feat:>15s}: {corr:+.4f}")

## Walk-Forward Fold Metrics & Gating Analysis

Per-fold performance, gating weights evolution, ROC curves, confusion matrices, and prediction analysis.

**What to look at:** fold AUC spread (0.43â€“0.60) is part of the finding â€” no consistent edge. Gate table: Î² dominates, Î³â‰ˆ0 everywhere (news shut off by training, not by bug â€” see grad audit). Prediction timeline + confusion matrices: model leans DOWN at thr 0.5, hence Youden-J reporting.


In [ ]:
# Fold metrics bar chart
if wf_metrics and "fold_metrics" in wf_metrics:
    fm = wf_metrics["fold_metrics"]
    folds = [m["fold"] for m in fm]
    aucs = [m["auc"] for m in fm]
    accs = [m["acc"] for m in fm]
    f1s = [m["f1"] for m in fm]
    n_samples = [m["n_samples"] for m in fm]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    x = np.arange(len(folds))
    width = 0.25
    ax1.bar(x - width, aucs, width, label="AUC", color="steelblue", alpha=0.8)
    ax1.bar(x, accs, width, label="Accuracy", color="orange", alpha=0.8)
    ax1.bar(x + width, f1s, width, label="F1", color="green", alpha=0.8)
    ax1.set_xlabel("Fold")
    ax1.set_ylabel("Score")
    ax1.set_title("Walk-Forward Fold Metrics")
    ax1.set_xticks(x)
    ax1.set_xticklabels([f"Fold {f}" for f in folds])
    ax1.legend()
    ax1.set_ylim(0, 1.05)
    ax1.grid(axis="y", alpha=0.3)

    # Sample counts
    ax2.bar(folds, n_samples, color="gray", alpha=0.6)
    ax2.set_xlabel("Fold")
    ax2.set_ylabel("Samples")
    ax2.set_title("Samples per Fold")
    ax2.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print summary
    print("Fold  | AUC    | Acc    | F1     | Samples")
    print("------|--------|--------|--------|--------")
    for m in fm:
        print(
            f"  {m['fold']}   | {m['auc']:.4f} | {m['acc']:.4f} | {m['f1']:.4f} | {m['n_samples']}"
        )
    avg_auc = np.mean([m["auc"] for m in fm])
    avg_acc = np.mean([m["acc"] for m in fm])
    avg_f1 = np.mean([m["f1"] for m in fm])
    print(f"Mean  | {avg_auc:.4f} | {avg_acc:.4f} | {avg_f1:.4f}")
else:
    print("No fold metrics available for visualization.")

In [ ]:
# Gating weights (Î±, Î², Î³) evolution over folds
if wf_metrics and "weights_list" in wf_metrics:
    weights = np.array(wf_metrics["weights_list"])  # shape: (n_folds, 3)
    folds = wf_metrics.get("fold_indices", list(range(len(weights))))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Weights over folds
    ax1.plot(folds, weights[:, 0], "o-", label="Î± (Technical)", color="steelblue", markersize=8)
    ax1.plot(folds, weights[:, 1], "s-", label="Î² (Macro)", color="orange", markersize=8)
    ax1.plot(folds, weights[:, 2], "^-", label="Î³ (News)", color="green", markersize=8)
    ax1.set_xlabel("Fold")
    ax1.set_ylabel("Weight")
    ax1.set_title("Gating Weights (Î±, Î², Î³) Evolution")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1.05)

    # Stacked area for composition
    ax2.stackplot(
        folds,
        weights[:, 0],
        weights[:, 1],
        weights[:, 2],
        labels=["Î± (Technical)", "Î² (Macro)", "Î³ (News)"],
        colors=["steelblue", "orange", "green"],
        alpha=0.7,
    )
    ax2.set_xlabel("Fold")
    ax2.set_ylabel("Weight")
    ax2.set_title("Gating Weights Composition")
    ax2.legend(loc="upper right")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("Fold| Î± (Tech) | Î² (Macro)  | Î³ (News)")
    print("----|----------|------------|---------")
    for i, w in enumerate(weights):
        print(f"  {folds[i]} | {w[0]:.4f}   | {w[1]:.4f}     | {w[2]:.4f}")
    avg_w = weights.mean(axis=0)
    print(f"Mean| {avg_w[0]:.4f}   | {avg_w[1]:.4f}     | {avg_w[2]:.4f}")
else:
    print("No gating weights available for visualization.")

In [ ]:
# Prediction vs Actual timeline
if wf_metrics and "preds" in wf_metrics:
    preds = wf_metrics["preds"]
    truths = wf_metrics["truths"]
    fold_indices = wf_metrics.get("fold_indices", list(range(len(preds))))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    x = np.arange(len(preds))

    # Predictions with actual markers
    ax1.plot(x, preds, "o-", color="steelblue", label="Predicted Probability", markersize=6)
    ax1.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="Decision Threshold")
    # Mark actual labels
    up_idx = [i for i, t in enumerate(truths) if t == 1.0]
    down_idx = [i for i, t in enumerate(truths) if t == 0.0]
    ax1.scatter(
        up_idx,
        [preds[i] for i in up_idx],
        color="green",
        s=80,
        marker="^",
        label="Actual UP",
        zorder=5,
    )
    ax1.scatter(
        down_idx,
        [preds[i] for i in down_idx],
        color="red",
        s=80,
        marker="v",
        label="Actual DOWN",
        zorder=5,
    )
    # Fold boundaries
    if "fold_indices" in wf_metrics and len(wf_metrics["fold_indices"]) > 1:
        for fi in wf_metrics["fold_indices"]:
            if fi < len(preds):
                ax1.axvline(fi, color="gray", linestyle=":", alpha=0.5)
    ax1.set_ylabel("Predicted Probability")
    ax1.set_title("Walk-Forward Predictions vs Actual Labels")
    ax1.legend(loc="upper right", fontsize=8)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(-0.05, 1.05)

    # Accuracy over time (rolling)
    window = max(1, len(preds) // 5)
    rolling_acc = (pd.Series(np.asarray(preds) > 0.5) == pd.Series(truths)).rolling(window, min_periods=1).mean().values
    ax2.plot(
        x, rolling_acc, "o-", color="orange", label=f"Rolling Acc (window={window})", markersize=4
    )
    ax2.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
    ax2.set_xlabel("Fold / Sample Index")
    ax2.set_ylabel("Rolling Accuracy")
    ax2.set_title("Rolling Prediction Accuracy")
    ax2.legend(loc="lower right", fontsize=8)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 1.05)

    plt.tight_layout()
    plt.show()
else:
    print("No predictions available for visualization.")

In [ ]:
# ROC curves per fold
if wf_metrics and "fold_metrics" in wf_metrics:
    fm = wf_metrics["fold_metrics"]
    fig, ax = plt.subplots(figsize=(6, 6))

    for m in fm:
        if "preds" in m and "truths" in m and len(set(m["truths"])) > 1:
            fpr, tpr, _ = roc_curve(m["truths"], m["preds"])
            fold_auc = m["auc"]
            ax.plot(fpr, tpr, "o-", label=f"Fold {m['fold']} (AUC={fold_auc:.3f})", markersize=4)

    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curves per Walk-Forward Fold")
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal")

    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data for ROC curves.")

In [ ]:
if wf_metrics and "fold_metrics" in wf_metrics:
    fm = wf_metrics["fold_metrics"]
    n_folds = len(fm)
    fig, axes = plt.subplots(1, n_folds, figsize=(4 * n_folds, 3))
    if n_folds == 1:
        axes = [axes]

    for idx, m in enumerate(fm):
        ax = axes[idx]
        pred_labels = [1 if p > 0.5 else 0 for p in m["preds"]]
        truths = m["truths"]
        cm = confusion_matrix(truths, pred_labels, labels=[0, 1])
        im = ax.imshow(cm, interpolation="nearest", cmap="Blues", vmin=0, vmax=max(cm.max(), 1))
        ax.set_title(f"Fold {m['fold']} (Acc={m['acc']:.3f})")
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(["DOWN", "UP"])
        ax.set_yticklabels(["DOWN", "UP"])
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        # Annotate
        for i in range(2):
            for j in range(2):
                ax.text(
                    j,
                    i,
                    str(cm[i, j]),
                    ha="center",
                    va="center",
                    fontsize=14,
                    color="white" if cm[i, j] > cm.max() / 2 else "black",
                )
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()
else:
    print("No fold data for confusion matrices.")

In [ ]:
# Gating weights distribution histograms
if wf_metrics and "weights_list" in wf_metrics:
    weights = np.array(wf_metrics["weights_list"])

    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    labels = ["Î± (Technical)", "Î² (Macro)", "Î³ (News)"]
    colors = ["steelblue", "orange", "green"]

    for i, (ax, label, color) in enumerate(zip(axes, labels, colors)):
        ax.hist(weights[:, i], bins=10, color=color, alpha=0.7, edgecolor="black")
        ax.axvline(
            weights[:, i].mean(),
            color="red",
            linestyle="--",
            label=f"Mean={weights[:, i].mean():.3f}",
        )
        ax.axvline(1 / 3, color="gray", linestyle=":", alpha=0.5, label="Uniform (1/3)")
        ax.set_xlabel("Weight")
        ax.set_ylabel("Frequency")
        ax.set_title(label)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.suptitle("Gating Weights Distribution Across Folds")
    plt.tight_layout()
    plt.show()

    # Statistics
    print("     Weight  | Mean   | Std    | Min    | Max")
    print("-------------|--------|--------|--------|---------")
    for i, label in enumerate(["Î± (IHSG)", "Î² (Macro)", "Î³ (News)"]):
        w = weights[:, i]
        print(f"{label:12s} | {w.mean():.4f} | {w.std():.4f} | {w.min():.4f} | {w.max():.4f}")
else:
    print("No gating weights for distribution visualization.")

In [ ]:
# Prediction confidence histogram
if wf_metrics and "preds" in wf_metrics:
    preds = np.array(wf_metrics["preds"])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    # Overall distribution
    ax1.hist(preds, bins=15, color="steelblue", alpha=0.7, edgecolor="black")
    ax1.axvline(0.5, color="red", linestyle="--", label="Decision Threshold (0.5)")
    ax1.axvline(preds.mean(), color="orange", linestyle="--", label=f"Mean={preds.mean():.3f}")
    ax1.set_xlabel("Predicted Probability")
    ax1.set_ylabel("Frequency")
    ax1.set_title("Prediction Confidence Distribution")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # By actual class
    truths = np.array(wf_metrics["truths"])
    up_preds = preds[truths == 1.0]
    down_preds = preds[truths == 0.0]

    ax2.hist(
        down_preds,
        bins=10,
        alpha=0.6,
        label=f"Actual DOWN (n={len(down_preds)})",
        color="red",
        edgecolor="black",
    )
    ax2.hist(
        up_preds,
        bins=10,
        alpha=0.6,
        label=f"Actual UP (n={len(up_preds)})",
        color="green",
        edgecolor="black",
    )
    ax2.axvline(0.5, color="red", linestyle="--", alpha=0.7, label="Threshold")
    ax2.set_xlabel("Predicted Probability")
    ax2.set_ylabel("Frequency")
    ax2.set_title("Prediction Distribution by Actual Class")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Statistics
    print(f"Overall: mean={preds.mean():.4f}, std={preds.std():.4f}")
    print(
        f"DOWN actual: n={len(down_preds)}, mean={down_preds.mean():.4f}, std={down_preds.std():.4f}"
    )
    print(f"UP actual:   n={len(up_preds)}, mean={up_preds.mean():.4f}, std={up_preds.std():.4f}")
    # Calibration: fraction of UP actual above 0.5
    if len(up_preds) > 0:
        print(f"UP recall (pred>0.5): {(up_preds > 0.5).mean():.4f}")
    if len(down_preds) > 0:
        print(f"DOWN recall (pred<0.5): {(down_preds < 0.5).mean():.4f}")
else:
    print("No predictions for confidence visualization.")

In [ ]:
# Reliability diagram (10 equal-width bins, pooled walk-forward preds) â€” read-only.
if wf_metrics and "preds" in wf_metrics:
    _p = np.array(wf_metrics["preds"])
    _t = np.array(wf_metrics["truths"]).astype(float)
    _eb = np.clip(np.digitize(_p, np.linspace(0, 1, 11)) - 1, 0, 9)
    _conf, _acc, _w = [], [], []
    for b in range(10):
        _m = _eb == b
        if _m.any():
            _conf.append(_p[_m].mean()); _acc.append(_t[_m].mean()); _w.append(_m.mean())
    _conf, _acc, _w = map(np.array, (_conf, _acc, _w))
    _ece = float((np.abs(_conf - _acc) * _w).sum())
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect")
    ax.plot(_conf, _acc, "o-", color="steelblue", label="Model")
    for c, a, w in zip(_conf, _acc, _w):
        ax.scatter([c], [a], s=400 * w, color="steelblue", alpha=0.3)  # size = bin mass
    ax.set_xlabel("Mean predicted prob"); ax.set_ylabel("Fraction UP")
    ax.set_title(f"Reliability (ECE={_ece:.4f})"); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f"ECE={_ece:.4f} (dot size = bin mass)")
else:
    print("(reliability skipped â€” run the pipeline cell first)")


In [ ]:
# Per-portal news attribution (descriptive â€” no retraining, architecture untouched).
if "news" in dir() and len(news):
    for _src, _g in news.groupby("src"):
        _v = np.stack(_g["vec"].values)
        print(f"{_src:>7}: n={len(_g):>6} days={_g['day'].min().date()}->{_g['day'].max().date()} mean|vec|={np.abs(_v).mean():.4f}")
    _daily = news.groupby([news["day"], news["src"]]).size().unstack(fill_value=0)
    fig, ax = plt.subplots(figsize=(10, 3))
    for _c in _daily.columns:
        ax.plot(_daily.index, _daily[_c], label=_c, linewidth=1)
    ax.legend(fontsize=8); ax.set_title("Articles per day by portal")
    plt.tight_layout(); plt.show()
else:
    print("(portal attribution skipped â€” run the pipeline cell first)")


## Gate adaptivity, regime split & calibration

Behavioral evidence for RQ3 (does the gate follow volatility?), COVID vs rest, and ECE â€” read-only, no retraining.


In [ ]:
# RQ3 behavior check: does the gate co-move with market volatility (ATR-14)?
# Fold-level only (n=5) â€” indicative, not proof. Read-only over stored outputs.
if wf_metrics and "weights_list" in wf_metrics and wf_metrics.get("test_dates"):
    _w = np.array(wf_metrics["weights_list"])
    _td = pd.to_datetime(pd.Series(wf_metrics["test_dates"]))
    _atr = df.set_index(pd.to_datetime(df["Date"]))["ATR_14"]
    _folds = wf_metrics["fold_metrics"]
    _atr_m, _s = [], 0
    for _m in _folds:
        _n = int(_m["n_samples"]); _atr_m.append(_atr.reindex(_td.iloc[_s:_s + _n]).mean()); _s += _n
    _atr_m = np.array(_atr_m, dtype=float)
    print("fold| mean ATR-14 | Î± | Î² | Î³")
    for _k, _m in enumerate(_folds):
        print(f"  {_k} | {_atr_m[_k]:.1f} | {_w[_k,0]:.3f} | {_w[_k,1]:.3f} | {_w[_k,2]:.3f}")
    print(f"corr(ATR,Î±)={np.corrcoef(_atr_m,_w[:,0])[0,1]:+.2f} corr(ATR,Î²)={np.corrcoef(_atr_m,_w[:,1])[0,1]:+.2f} corr(ATR,Î³)={np.corrcoef(_atr_m,_w[:,2])[0,1]:+.2f} (n=5 folds)")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(_atr_m, _w[:,1], s=80, label="Î² (Macro)", color="seagreen")
    ax.scatter(_atr_m, _w[:,0], s=80, marker="^", label="Î± (Technical)", color="steelblue")
    ax.set_xlabel("Mean test-period ATR-14"); ax.set_ylabel("Gate weight"); ax.legend()
    ax.set_title("Gate weights vs volatility (per fold)")
    plt.tight_layout(); plt.show()
else:
    print("(gate-vs-ATR skipped â€” re-run the pipeline cell to store test_dates)")


In [ ]:
# Regime split (COVID 2020-21 vs rest). Read-only over stored outputs. ECE lives in the reliability cell.
if wf_metrics and wf_metrics.get("test_dates") and "preds" in wf_metrics:
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
    _p = np.array(wf_metrics["preds"]); _t = np.array(wf_metrics["truths"])
    _yr = pd.to_datetime(pd.Series(wf_metrics["test_dates"])).dt.year.values
    for _name, _mask in [("COVID 2020-21", np.isin(_yr, [2020, 2021])), ("rest", ~np.isin(_yr, [2020, 2021]))]:
        _pm, _tm = _p[_mask], _t[_mask]
        _auc = float(roc_auc_score(_tm, _pm)) if len(set(_tm.tolist())) > 1 else float("nan")
        print(f"{_name}: n={_mask.sum()} AUC={_auc:.4f} Acc={accuracy_score(_tm, _pm > 0.5):.4f} F1={f1_score(_tm, _pm > 0.5, zero_division=0):.4f}")
    print(f"mean pred={_p.mean():.4f}")
else:
    print("(regime skipped â€” re-run the pipeline cell to store test_dates)")


## Trading equity & per-class precision

Docx Bab 4 demands the portfolio line chart: equity needs a re-run (series newly stored); precision prints on any run.


In [ ]:
# Trading equity curves & per-class precision (all variants).
# Equity series stored in wf_metrics["ablation"][v]["trading"]["strategy"]["equity"].
if wf_metrics and "preds" in wf_metrics and "ablation" in wf_metrics:
    from sklearn.metrics import precision_score

    # --- Per-class precision table (thr=0.5) ---
    print(f"{'Model':>18} | {'Prec DOWN':>10} | {'Prec UP':>8} | {'Sharpe (strat/bh)':>20} | {'Max DD':>8} | {'Win Rate (strat/bh)':>20}")
    print("-" * 95)

    _ab = wf_metrics["ablation"]
    _variants = ["full", "lstm", "lstm_macro", "lstm_news"]  # matching thesis Table 6 order
    _colors = {"full": "darkorange", "lstm": "steelblue", "lstm_macro": "seagreen", "lstm_news": "purple"}

    _eq_data = {}
    for _v in _variants:
        if _v not in _ab:
            continue
        _tr = _ab[_v]["trading"]["strategy"]
        _bh = _ab[_v]["trading"]["buyhold"]

        # Precision
        _p = (np.array(_ab[_v]["preds"]) > 0.5).astype(int)
        _t = np.array(_ab[_v]["truths"]).astype(int)
        _pr = precision_score(_t, _p, average=None, labels=[0, 1], zero_division=0)

        _eq_data[_v] = _tr
        print(f"{_v:>18} | {_pr[0]:10.4f} | {_pr[1]:8.4f} | {_tr['sharpe']:6.3f}/{_bh['sharpe']:6.3f} | {_tr['max_drawdown']:8.3f} | {_tr['win_rate']:6.3f}/{_bh['win_rate']:6.3f}")

    # --- Equity curves ---
    _first_v = next(v for v in _variants if v in _eq_data)
    _bh = _ab[_first_v]["trading"]["buyhold"]
    _n = min([len(_eq_data[v]["equity"]) for v in _eq_data] + [len(_bh["equity"])])
    _curves = [("Buy & hold", _bh, "#666666")]
    _display_names = {"full": "Full", "lstm": "LSTM", "lstm_macro": "LSTM + Macro", "lstm_news": "LSTM + News"}
    _curves.extend((_display_names.get(v, v.replace("_", " ").title()), _eq_data[v], _colors.get(v, "black"))
                   for v in _variants if v in _eq_data)
    _values = [np.asarray(stats["equity"][:_n], dtype=float) for _, stats, _ in _curves]
    _low = min(float(np.nanmin(values)) for values in _values)
    _high = max(float(np.nanmax(values)) for values in _values)
    _pad = max((_high - _low) * 0.08, 0.02)

    fig = plt.figure(figsize=(14, 6))
    gs = fig.add_gridspec(1, 2, width_ratios=[3.8, 1.7], wspace=0.06)
    ax = fig.add_subplot(gs[0, 0])
    ax_summary = fig.add_subplot(gs[0, 1])
    _x = np.arange(_n)
    for _name, _stats, _color in _curves:
        _equity = np.asarray(_stats["equity"][:_n], dtype=float)
        _is_bh = _name == "Buy & hold"
        ax.plot(_x, _equity, label=_name, color=_color,
                linewidth=2.2 if _name == "Full" else (1.8 if _is_bh else 1.5),
                linestyle="--" if _is_bh else "-", alpha=0.9 if _is_bh else 0.95,
                zorder=3 if _name == "Full" else (2 if _is_bh else 1))
    ax.axhline(1.0, color="#888888", linewidth=0.9, linestyle=":", alpha=0.8)
    ax.set_ylim(_low - _pad, _high + _pad)
    ax.set_xlim(0, _n - 1)
    ax.set_xlabel("Trading day in test window")
    ax.set_ylabel("Portfolio value (start = 1.00)")
    ax.grid(axis="y", alpha=0.25)
    for _spine in ("top", "right"):
        ax.spines[_spine].set_visible(False)

    ax_summary.axis("off")
    _rows = [[name, f"{stats['equity'][_n - 1]:.2f}x", f"{stats['max_drawdown']:.1%}"]
             for name, stats, _ in _curves]
    _table = ax_summary.table(cellText=_rows, colLabels=["Variant", "End", "Max DD"],
                              colWidths=[0.46, 0.24, 0.30], cellLoc="center", colLoc="center",
                              bbox=[0, 0.2, 1, 0.68])
    _table.auto_set_font_size(False)
    _table.set_fontsize(8)
    for _cell in _table.get_celld().values():
        _cell.set_edgecolor("#dddddd")
        _cell.set_linewidth(0.6)
    for _col in range(3):
        _table[(0, _col)].set_facecolor("#edf1f5")
        _table[(0, _col)].get_text().set_fontweight("bold")
    for _row, (_, _, _color) in enumerate(_curves, start=1):
        _table[(_row, 0)].get_text().set_color(_color)
        _table[(_row, 0)].get_text().set_fontweight("bold")
    ax_summary.set_title("Ending performance", fontsize=10, fontweight="bold", pad=12)
    fig.suptitle("Walk-forward equity | long/short vs buy & hold (no costs)", fontsize=13, fontweight="bold", y=0.98)
    fig.legend(*ax.get_legend_handles_labels(), loc="upper center", bbox_to_anchor=(0.48, 0.93),
               ncol=3, fontsize=8, frameon=False)
    fig.subplots_adjust(top=0.84, bottom=0.12, left=0.08, right=0.98)
    plt.show()

    # --- Per-class precision @ Youden-J threshold ---
    print("\nPer-class precision @ mean train Youden-J threshold:")
    for _v in _variants:
        if _v not in _ab:
            continue
        _tj = float(np.mean([m["thr"] for m in _ab[_v]["fold_metrics"]]))
        _p = (np.array(_ab[_v]["preds"]) > _tj).astype(int)
        _t = np.array(_ab[_v]["truths"]).astype(int)
        _pr = precision_score(_t, _p, average=None, labels=[0, 1], zero_division=0)
        print(f"  {_v:>18}: DOWN={_pr[0]:.4f} UP={_pr[1]:.4f} (thr={_tj:.4f})")
else:
    print("(equity/precision skipped -- run the pipeline cell first)")

In [ ]:
# Trading at mean train Youden-J threshold vs 0.5 (full variant, read-only demo).
if wf_metrics and "preds" in wf_metrics and "test_dates" in wf_metrics:
    _tj = float(np.mean([m["thr"] for m in wf_metrics["fold_metrics"]]))
    _b = trading_stats(wf_metrics["test_dates"], wf_metrics["preds"], df, thr=0.5)["strategy"]
    _y = trading_stats(wf_metrics["test_dates"], wf_metrics["preds"], df, thr=_tj)["strategy"]
    print(f"mean train Youden-J thr={_tj:.4f}")
    print(f"thr 0.5     : Sharpe={_b['sharpe']:.3f} MDD={_b['max_drawdown']:.4f} win={_b['win_rate']:.4f}")
    print(f"thr Youden-J: Sharpe={_y['sharpe']:.3f} MDD={_y['max_drawdown']:.4f} win={_y['win_rate']:.4f}")
else:
    print("(youden trading demo skipped â€” run the pipeline cell first)")


In [ ]:
# News aggregation input: what daily mean-pooling aggregates â€” read-only, DB-backed.
if "news" in dir() and len(news):
    _cnt = pd.to_datetime(news["Date"].dt.date).value_counts()
    if "df" in dir() and "Date" in df.columns:
        _all = pd.to_datetime(pd.Series(sorted(set(df["Date"].dt.date))))
        _per = _all.map(_cnt).fillna(0).astype(int); _scope = "trading days"
    else:
        _per = _cnt.sort_index().reset_index(drop=True); _scope = "article-days"
    print(f"days ({_scope}): {len(_per)}, with >=1 article: {(_per > 0).mean():.1%}, max/day: {_per.max()}, mean/day: {_per.mean():.2f}")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))
    ax1.hist(_per.values, bins=range(int(_per.max()) + 2), color="seagreen", alpha=0.7, edgecolor="black")
    ax1.set_xlabel("articles per day"); ax1.set_ylabel("days"); ax1.set_title("Daily article-count distribution")
    ax1.axvline(1, color="red", linestyle="--", linewidth=0.8)
    _by = news.groupby("src").size()
    ax2.bar(_by.index, _by.values, color=["steelblue", "orange", "green"][:len(_by)])
    ax2.set_title("Articles by portal"); ax2.set_ylabel("count")
    for _x, _v in zip(_by.index, _by.values): ax2.text(_x, _v, str(_v), ha="center", va="bottom", fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print("(news aggregation viz skipped â€” run the pipeline cell first)")


## Checks

Shapes, finiteness, binary labels, and proof the scalers saw the train prefix only.

**What to look at:** two OK lines = architecture matches Table 5 AND scalers saw the 70% train prefix only. If either fails, nothing downstream is trustworthy.


In [ ]:
assert X_tech.shape[1:] == (lb, len(CFG.tech_cols))
assert len(y) == n_win
assert np.isfinite(X_tech).all() and np.isfinite(X_macro).all()
assert set(np.unique(y)) <= {0.0, 1.0}
raw_tech = df[list(CFG.tech_cols)].values.astype(np.float32)
assert np.allclose(feat_scaler.data_min_, raw_tech[:train_end_df].min(axis=0))

model = StockModel(CFG)  # rebuild so arch asserts actually run (run cell leaves model=None)
assert all(hasattr(model, a) for a in ("lstm", "mlp", "news_proj", "gate"))
assert model.lstm.hidden_size == CFG.lstm_dim
assert model.mlp[-2].out_features == CFG.hidden_layers[-1]
assert model.news_proj.out_features == CFG.lstm_dim
assert model.gate.out_features == CFG.fusion_output_dim
assert model.gate.in_features == CFG.lstm_dim + CFG.hidden_layers[-1] + CFG.lstm_dim
assert "auc" in wf_metrics and "acc" in wf_metrics and "f1" in wf_metrics
# inline evaluate_model for single-sample check (using df.iloc[:1])
model.eval()
with torch.no_grad():
    sub_df = df.iloc[:1]
    lookback = CFG.look_back
    tech_cols = [c for c in sub_df.columns if c in CFG.tech_cols]
    tech_seq = sub_df[tech_cols].values.astype(np.float32)
    macro_seq = sub_df[CFG.macro_cols].values.astype(np.float32)
    if len(tech_seq) < lookback:
        prob = 0.5
    else:
        news_seq = sub_df[NEWS_COLS].values.astype(np.float32)
        tech_batch = torch.tensor(tech_seq[-lookback:].reshape(1, lookback, -1), dtype=torch.float32)
        macro_batch = torch.tensor(macro_seq[-1:].reshape(1, -1), dtype=torch.float32)
        news_batch = torch.tensor(news_seq[-lookback:].mean(axis=0, keepdims=True), dtype=torch.float32)
        logits, _, _ = model(tech_batch, macro_batch, news_batch)
        prob = torch.sigmoid(logits).item()
assert prob == 0.5
print("OK: thesis model architecture verified")

print("OK: leak-free single-notebook run passed")


## Summary Report

Thesis defense results summary: model architecture, walk-forward metrics, gating weights, and key findings.

**What to look at:** Key Findings lines 1â€“4 are the thesis-ready sentences (dominant modality, pooled AUC, training regime, live news counts). UP/DOWN recall explains the Youden-J choice.


In [ ]:
"""Thesis Defense Summary Report"""
print("=" * 70)
print("MSDL-JCI: Multi-Source Deep Learning for JCI Direction Prediction")
print("Thesis Defense Summary Report")
print("=" * 70)

# Model Architecture Summary (thesis Table 5)
# live news counts (corpus changes across scrapes â€” never hardcode)
try:
    _jmax = pd.to_datetime(frames[0]["Date"], format="mixed", errors="coerce").max()
    _nline = (f"4. News: frozen indobert-base-p1 CLS, {len(news)} articles "
              f"{dict(news['src'].value_counts())}, "
              f"{(news['day'] <= _jmax).sum()} on/before JCI end past-only")
except (OSError, RuntimeError):
    _nline = "4. News: frozen indobert-base-p1 CLS (re-run pipeline cell for counts)"
for line in [
    f"Technical: LSTM(hidden={CFG.lstm_dim}, lookback={CFG.look_back}, dropout=0.2)",
    f"Macro:     MLP({len(CFG.macro_cols)}â†’{'â†’'.join(map(str, CFG.hidden_layers))}, ReLU)",
    f"News:      Frozen IndoBERT(768â†’{CFG.projection_dim})",
    f"Fusion:    SoftGate({CFG.lstm_dim}+{CFG.hidden_layers[-1]}+{CFG.projection_dim}â†’{CFG.fusion_output_dim})",
    f"Head:      Linear({CFG.lstm_dim}â†’1, BCE)",
]:
    print(line)

# Data Summary
print(f"Rows={len(df)}  Samples={len(y)}  Range={dates[0]}â†’{dates[-1]}  UP={y.mean():.1%}/DOWN={1-y.mean():.1%}")

# Walk-Forward Results
if wf_metrics and "fold_metrics" in wf_metrics:
    fm = wf_metrics["fold_metrics"]
    print("\n--- Walk-Forward Results (5 splits, expanding window) ---")
    print(f"{'Fold':>4} | {'AUC':>6} | {'Acc':>6} | {'F1':>6} | {'N':>4}")
    print("-" * 40)
    for m in fm:
        print(f"{m['fold']:>4} | {m['auc']:>6.4f} | {m['acc']:>6.4f} | {m['f1']:>6.4f} | {m['n_samples']:>4}")
    mean_auc = np.mean([m['auc'] for m in fm])
    mean_acc = np.mean([m['acc'] for m in fm])
    mean_f1 = np.mean([m['f1'] for m in fm])
    print("-" * 40)
    print(f"{'Mean':>4} | {mean_auc:>6.4f} | {mean_acc:>6.4f} | {mean_f1:>6.4f} |")

    # Gating weights summary
    if "weights_list" in wf_metrics:
        weights = np.array(wf_metrics["weights_list"])
        print("\n--- Gate Weights (Î±, Î², Î³) Summary ---")
        print(f"Technical (Î±): mean={weights[:,0].mean():.4f}, std={weights[:,0].std():.4f}")
        print(f"Macro (Î²):     mean={weights[:,1].mean():.4f}, std={weights[:,1].std():.4f}")
        print(f"News (Î³):      mean={weights[:,2].mean():.4f}, std={weights[:,2].std():.4f}")

    # Visualization: summary heatmap
    fig, ax1 = plt.subplots(figsize=(8, 5))

    # Left: fold metrics heatmap
    fold_data = np.array([[m["auc"], m["acc"], m["f1"]] for m in fm])
    im1 = ax1.imshow(fold_data, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
    ax1.set_xticks([0, 1, 2])
    ax1.set_xticklabels(["AUC", "Accuracy", "F1"])
    ax1.set_yticks(range(len(fm)))
    ax1.set_yticklabels([f"Fold {m['fold']}" for m in fm])
    ax1.set_title("Walk-Forward Fold Metrics")
    for i in range(len(fm)):
        for j in range(3):
            ax1.text(j, i, f"{fold_data[i,j]:.3f}", ha="center", va="center", fontsize=9)
    plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()
else:
    print("No walk-forward results available.")

# Key Findings (guarded: summary must never crash if pipeline cell was skipped)
if not (wf_metrics and "weights_list" in wf_metrics and "preds" in wf_metrics):
    print("\n--- Key Findings ---")
    print("(skipped â€” run the pipeline cell first)")
else:
    w = np.array(wf_metrics["weights_list"]).mean(axis=0)
    dom = ["Technical (Î±)", "Macro (Î²)", "News (Î³)"][int(np.argmax(w))]
    for line in [
        f"1. Dominant modality: {dom} (Î±={w[0]:.3f}, Î²={w[1]:.3f}, Î³={w[2]:.3f})",
        f"2. Walk-forward AUC={wf_metrics['auc']:.4f}, Acc={wf_metrics['acc']:.4f}, F1={wf_metrics['f1']:.4f}",
        "3. Trained per fold (Adam, weighted BCE, early stopping max100/patience8, TSS n=5)",
        _nline,
    ]:
        print(line)
    preds = np.array(wf_metrics["preds"]) > 0.5
    truths = np.array(wf_metrics["truths"]) == 1
    print(f"UP recall={preds[truths].mean():.4f}, DOWN recall={(~preds[~truths]).mean():.4f}")
print("=" * 70)
